In [11]:
# Install the required libraries

!pip install requests pandas snowflake-connector-python

In [12]:
# Import Python's system module
import sys

# Display Python version
print("Python Version:", sys.version)

# Display Python executable
print("Python Executable:", sys.executable)

Python Version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Python Executable: /usr/bin/python3


In [13]:
# Check important libraries

import pandas as pd
import requests
import json
import os

In [14]:
# Create a project directory

project_folder = "FreshCart"

os.makedirs(project_folder, exist_ok=True)

print("Project directory created successfully!")

Project directory created successfully!


In [15]:
!pip install -q snowflake-connector-python

In [16]:
# ==========================================
# SINGLE SNOWFLAKE CONNECTION FOR FRESHCART
# Keep this connection before SQL_SETUP because
# the setup cell uses this connection to create
# the warehouse, database, schemas, and tables.
# ==========================================

from getpass import getpass
import snowflake.connector

SNOWFLAKE_ACCOUNT = "GP04529.ap-southeast-7.aws"
SNOWFLAKE_USER = "SONUSUMAN202"
SNOWFLAKE_ROLE = "ACCOUNTADMIN"

SNOWFLAKE_PASSWORD = getpass("Enter your Snowflake password: ")

# Local landing directory used by the extraction/loading stages
LANDING_DIR = "/content/freshcart_landing"

# Do not specify warehouse/database here: SQL_00_SETUP creates them.
conn = snowflake.connector.connect(
    account=SNOWFLAKE_ACCOUNT,
    user=SNOWFLAKE_USER,
    password=SNOWFLAKE_PASSWORD,
    role=SNOWFLAKE_ROLE
)

print("Successfully connected to Snowflake!")

with conn.cursor() as cur:
    cur.execute("SELECT CURRENT_USER(), CURRENT_ROLE()")
    print("Connection details:", cur.fetchone())


Enter your Snowflake password: ··········


Successfully connected to Snowflake!
Connection details: ('SONUSUMAN202', 'ACCOUNTADMIN')


In [74]:
# Define deliberately messy sample API data so we can demonstrate cleaning, deduplication, and enrichment.
# ---- Website orders: flat JSON ----
MOCK_WEB_ORDERS = [
    # ORD-1001: email has spaces and capitals, amount has a comma
    {"order_id": "ORD-1001", "cust_email": " Priya.R@Gmail.com ", "cust_name": "priya r",
     "order_date": "2026-09-01", "amount": "1,250.00", "currency": "INR", "pincode": "600041",
     "updated_at": "2026-09-01 10:02:00"},
    # ORD-1003: amount is missing (the mobile copy below has it)
    {"order_id": "ORD-1003", "cust_email": "meera@outlook.com", "cust_name": "MEERA S",
     "order_date": "2026-09-02", "amount": None, "currency": "INR", "pincode": "400050",
     "updated_at": "2026-09-02 09:10:00"},
    # ORD-1002: a dollar order (also sent by the mobile app = duplicate)
    {"order_id": "ORD-1002", "cust_email": "Arjun.K@Yahoo.in ", "cust_name": "Arjun K",
     "order_date": "2026-09-01", "amount": "42.50", "currency": "USD", "pincode": "560034",
     "updated_at": "2026-09-01 11:15:00"},
    # ORD-1004: negative amount -> will be rejected
    {"order_id": "ORD-1004", "cust_email": "rahul.m@gmail.com", "cust_name": "Rahul M",
     "order_date": "Sep 3, 2026", "amount": "-500", "currency": "INR", "pincode": "110001",
     "updated_at": "2026-09-03 12:00:00"},
    # ORD-1005: rupee sign in the amount, lowercase currency, dot after the name
    {"order_id": "ORD-1005", "cust_email": "priya.r@gmail.com", "cust_name": "Priya R.",
     "order_date": "2026-09-03", "amount": "₹ 899", "currency": "inr", "pincode": "600041",
     "updated_at": "2026-09-03 16:45:00"},
    # ORD-1006: bad email and 5-digit pincode -> will be rejected
    {"order_id": "ORD-1006", "cust_email": "vikram@@mail", "cust_name": "Vikram",
     "order_date": "2026-09-04", "amount": 30, "currency": "EUR", "pincode": "99999",
     "updated_at": "2026-09-04 08:30:00"},
]

# ---- Mobile app orders: nested JSON, different field names, day-first dates ----
MOCK_MOBILE_ORDERS = [
    # ORD-1002 again (same as the web copy)
    {"orderId": "ORD-1002", "customer": {"email": "arjun.k@yahoo.in", "name": "Arjun K"},
     "orderDate": "01/09/2026", "total": {"value": "42.50", "currency": "usd"},
     "deliveryPin": "560034", "updatedAt": "2026-09-01T11:15:00"},
    # ORD-1003 again, newer and with the amount filled in
    {"orderId": "ORD-1003", "customer": {"email": "Meera@Outlook.com", "name": "Meera S"},
     "orderDate": "02/09/2026", "total": {"value": "2,100.00", "currency": "INR"},
     "deliveryPin": "400050", "updatedAt": "2026-09-02T18:40:00"},
]

# ---- Customer tier changes (Priya moves from Silver to Gold on 3 Sept) ----
MOCK_CRM_TIER_EVENTS = [
    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Silver", "effective_from": "2023-04-10"},
    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Gold", "effective_from": "2026-09-03"},
    {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in", "tier": "Silver", "effective_from": "2024-01-15"},
    {"cust_id": "CUST-03", "email": "meera@outlook.com", "tier": "Platinum", "effective_from": "2022-11-01"},
    {"cust_id": "CUST-02","email": "arjun.k@yahoo.in","tier": "Gold","effective_from": "2026-08-15"},
]

# ---- Exchange rates: rupees per 1 US dollar, per day ----
MOCK_FX_RESPONSES = [
    {"amount": 1.0, "base": "USD", "start_date": "2026-09-01", "end_date": "2026-09-03",
     "rates": {"2026-09-01": {"INR": 83.20}, "2026-09-02": {"INR": 83.35}, "2026-09-03": {"INR": 83.10}}},
]

# ---- Pincodes: pincode -> (city, state) ----
MOCK_PINCODES = {"600041": ("Chennai", "Tamil Nadu"), "560034": ("Bengaluru", "Karnataka"),
                 "400050": ("Mumbai", "Maharashtra")}


def mock_pincode_response(pin):
    """Reply in the same shape the India Post API uses."""
    if pin not in MOCK_PINCODES:
        return [{"Message": "No records found", "Status": "Error", "PostOffice": None}]
    district, state = MOCK_PINCODES[pin]
    return [{"Message": "Number of pincode(s) found:1", "Status": "Success",
             "PostOffice": [{"Name": district, "District": district, "State": state,
                             "Country": "India", "Pincode": pin}]}]

EXTRACT: save the data as raw files

In [18]:
# Define raw-file writing so each source is saved with run metadata before any transformations.
import gzip, json  # Import tools for gzip compression and JSON conversion.
from datetime import date, datetime, timezone  # Import date/time tools; date is unused here.
from pathlib import Path  # Import Path to work with folders and file paths.


def write_landing(source, records, run_id):  # Define a function to save records from one source.
    """Save source records in a compressed JSON Lines file and return its path."""
    folder = Path(LANDING_DIR) / source  # Build the source folder path; LANDING_DIR must already be defined.
    folder.mkdir(parents=True, exist_ok=True)  # Create the folder and missing parent folders if needed.
    path = folder / f"{source}__{run_id}.json.gz"  # Create a filename containing the source and run ID.
    now = datetime.now(timezone.utc).isoformat()  # Capture the current UTC extraction time.
    with gzip.open(path, "wt", encoding="utf-8") as f:  # Open a compressed file for writing UTF-8 text.
        for rec in records:  # Process each input record.
            envelope = {"source": source, "run_id": run_id, "extracted_at": now, "data": rec}  # Attach extraction metadata to the record.
            f.write(json.dumps(envelope, ensure_ascii=False) + "\n")  # Write one JSON record per line while preserving Unicode characters.
    print(f"   {source}: {len(records)} records -> {path}")  # Display the record count and saved file path.
    return path  # Return the saved file path.

In [19]:
# ============================================
# FRESHCART - STAGE 3.4: DATA EXTRACTION
# ============================================

from datetime import datetime, timezone

# Generate a unique ID for this pipeline execution
run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

print("FreshCart Extraction Started")
print("Run ID:", run_id)

# Extract and save website orders
web_file = write_landing(
    "web",
    MOCK_WEB_ORDERS,
    run_id
)

# Extract and save mobile orders
mobile_file = write_landing(
    "mobile",
    MOCK_MOBILE_ORDERS,
    run_id
)

# Extract and save customer tier events
crm_file = write_landing(
    "crm",
    MOCK_CRM_TIER_EVENTS,
    run_id
)

# Extract and save exchange rate information
fx_file = write_landing(
    "fx",
    MOCK_FX_RESPONSES,
    run_id
)

print("\nAll extraction operations completed!")

FreshCart Extraction Started
Run ID: 20261108_093354
   web: 6 records -> /content/freshcart_landing/web/web__20261108_093354.json.gz
   mobile: 2 records -> /content/freshcart_landing/mobile/mobile__20261108_093354.json.gz
   crm: 4 records -> /content/freshcart_landing/crm/crm__20261108_093354.json.gz
   fx: 1 records -> /content/freshcart_landing/fx/fx__20261108_093354.json.gz

All extraction operations completed!


In [20]:
# ============================================
# VERIFY RAW FILES
# ============================================

import os

print("FreshCart Landing Directory")
print("============================")

for root, directories, files in os.walk(LANDING_DIR):

    for file in files:

        file_path = os.path.join(root, file)

        file_size = os.path.getsize(file_path)

        print(f"File: {file}")
        print(f"Size: {file_size} bytes")
        print("----------------------------")

FreshCart Landing Directory
File: crm__20261108_093354.json.gz
Size: 292 bytes
----------------------------
File: fx__20261108_093354.json.gz
Size: 221 bytes
----------------------------
File: web__20261108_093354.json.gz
Size: 505 bytes
----------------------------
File: mobile__20261108_093354.json.gz
Size: 347 bytes
----------------------------


In [21]:
# Define SQL and file-loading helpers so the pipeline can upload raw files and inspect Snowflake tables.
import pandas as pd  # Import pandas to display query results as tables.


def run_sql(sql_text):  # Define a helper to execute multiple SQL statements.
    """Run a SQL script that contains several statements."""
    conn.execute_string(sql_text, remove_comments=True)  # Execute the SQL script after removing comments.


def query(sql):  # Define a helper to execute SQL and retrieve its results.
    """Run one query and return its rows."""
    with conn.cursor() as cur:  # Open a cursor that closes automatically when the block ends.
        cur.execute(sql)  # Execute the supplied SQL statement.
        return cur.fetchall()  # Return all result rows as a list of tuples.


def show(sql):  # Define a helper to return SQL results as a pandas table.
    """Run one query and return the result as a table."""
    with conn.cursor() as cur:  # Open a cursor to interact with Snowflake.
        cur.execute(sql)  # Execute the supplied query.
        return pd.DataFrame(
            cur.fetchall(),  # Retrieve all result rows.
            columns=[c[0] for c in cur.description]  # Use the query's column names as table headings.
        )


def load_file(local_path, table, stage_dir):  # Define a helper to upload a file and load its records.
    """Upload one raw file to Snowflake, then copy its records into a bronze table."""
    uri = "file://" + Path(local_path).resolve().as_posix()  # Convert the local path into an absolute file URI.
    query(
        f"PUT '{uri}' @bronze.landing/{stage_dir}/ AUTO_COMPRESS=FALSE OVERWRITE=FALSE"
    )  # Upload the file to the stage without additional compression or overwriting existing files.
    rows = query(f"""
        -- Load records from the specified stage folder into the target table.
        COPY INTO {table} (source, raw, run_id, extracted_at, src_file, src_row)
        FROM (
          -- Extract JSON fields and capture the source filename and row number.
          SELECT $1:source::STRING, $1:data, $1:run_id::STRING,
                 $1:extracted_at::TIMESTAMP_TZ, METADATA$FILENAME, METADATA$FILE_ROW_NUMBER
          -- Read staged files from this folder.
          FROM @bronze.landing/{stage_dir}/
        )
        -- Parse files using the predefined newline-delimited JSON format.
        FILE_FORMAT = (FORMAT_NAME = 'bronze.ff_ndjson')
        -- Abort the COPY statement if a loading error occurs.
        ON_ERROR = 'ABORT_STATEMENT'
    """)  # Execute COPY INTO and collect its loading results.
    loaded = sum(int(r[3]) for r in rows if len(r) > 3)  # Sum the rows loaded from the fourth column of each COPY result.
    print(f"   {loaded} rows -> {table}")  # Display the total loaded rows and destination table.

In [22]:
# Store setup SQL for later execution so the warehouse, schemas, stage, and tables are ready before loading.
SQL_00_SETUP = r"""
-- =====================================================================
-- Setup: creates the warehouse, database, schemas and empty tables.
-- Safe to re-run: everything is IF NOT EXISTS, except the two small seed tables.
-- =====================================================================
CREATE WAREHOUSE IF NOT EXISTS FRESHCART_WH
  WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE FRESHCART_WH;

CREATE DATABASE IF NOT EXISTS FRESHCART;
USE DATABASE FRESHCART;

CREATE SCHEMA IF NOT EXISTS bronze;      -- raw API responses, unchanged
CREATE SCHEMA IF NOT EXISTS silver;      -- cleaned, typed, deduplicated
CREATE SCHEMA IF NOT EXISTS dim;         -- dimensions (customers)
CREATE SCHEMA IF NOT EXISTS ref;         -- reference data (fx, pincodes, tiers)
CREATE SCHEMA IF NOT EXISTS gold;        -- business-ready tables
CREATE SCHEMA IF NOT EXISTS quarantine;  -- rows that failed validation

-- Landing area inside Snowflake for the files Python uploads
CREATE FILE FORMAT IF NOT EXISTS bronze.ff_ndjson TYPE = JSON COMPRESSION = AUTO;
CREATE STAGE IF NOT EXISTS bronze.landing FILE_FORMAT = bronze.ff_ndjson;

-- ---------- BRONZE: one table per source, raw JSON in a VARIANT column ----------
CREATE TABLE IF NOT EXISTS bronze.orders_raw (
  ingest_seq   NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER,   -- arrival order
  source       STRING,          -- 'web' or 'mobile'
  raw          VARIANT,         -- the API record exactly as received
  run_id       STRING,
  extracted_at TIMESTAMP_TZ,
  src_file     STRING,
  src_row      NUMBER,
  loaded_at    TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);
CREATE TABLE IF NOT EXISTS bronze.customers_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);
CREATE TABLE IF NOT EXISTS bronze.fx_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);
CREATE TABLE IF NOT EXISTS bronze.pincode_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

-- ---------- REF: small hand-maintained seed tables ----------
CREATE OR REPLACE TABLE ref.tier_multiplier AS
SELECT * FROM VALUES ('Silver', 1), ('Gold', 2), ('Platinum', 3) AS v(tier, multiplier);

CREATE OR REPLACE TABLE ref.state_region AS
SELECT * FROM VALUES
  ('Tamil Nadu','South'), ('Karnataka','South'), ('Kerala','South'), ('Andhra Pradesh','South'),
  ('Telangana','South'), ('Puducherry','South'),
  ('Maharashtra','West'), ('Gujarat','West'), ('Goa','West'), ('Rajasthan','West'),
  ('Delhi','North'), ('Haryana','North'), ('Punjab','North'), ('Uttar Pradesh','North'),
  ('Himachal Pradesh','North'), ('Jammu and Kashmir','North'), ('Uttarakhand','North'),
  ('West Bengal','East'), ('Odisha','East'), ('Bihar','East'), ('Jharkhand','East'), ('Assam','East'),
  ('Madhya Pradesh','Central'), ('Chhattisgarh','Central')
  AS v(state, region);

-- Filled incrementally from the India Post API (see SQL_32_PINCODE_GEO)
CREATE TABLE IF NOT EXISTS ref.pincode_geo (
  pincode STRING PRIMARY KEY, city STRING, state STRING, region STRING,
  updated_at TIMESTAMP_LTZ
);

-- ---------- QUARANTINE, GOLD ----------
CREATE TABLE IF NOT EXISTS quarantine.orders (
  run_id STRING, quarantined_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
  ingest_seq NUMBER, source STRING, order_id STRING, reasons ARRAY, row_data VARIANT
);

CREATE TABLE IF NOT EXISTS gold.fct_orders (
  order_id STRING PRIMARY KEY, order_date DATE, cust_id STRING, email STRING,
  tier STRING, multiplier NUMBER, amount NUMBER(12,2), currency STRING,
  fx_rate NUMBER(12,4), amount_inr NUMBER(14,2), loyalty_points NUMBER,
  is_high_value BOOLEAN, city STRING, state STRING, region STRING,
  updated_at TIMESTAMP_NTZ, row_hash NUMBER, published_at TIMESTAMP_LTZ
);
"""

In [23]:
# ==========================================
# STAGE 4.1: CREATE SNOWFLAKE OBJECTS
# ==========================================

run_sql(SQL_00_SETUP)

print("FreshCart Snowflake setup completed!")

FreshCart Snowflake setup completed!


In [66]:
# ================================================================
# STAGE 4.3 - CELL 1: ORDERS CLEANING AND VALIDATION
# ================================================================

# Store cleaning SQL for later execution so web/mobile fields
# are standardized and invalid orders are quarantined.

SQL_10_ORDERS_CLEAN = r"""

-- =====================================================================
-- SILVER 1: unify the two order APIs, cleanse, validate, quarantine.
-- Input : bronze.orders_raw (raw JSON from 'web' and 'mobile')
-- Output: silver.orders_checked (all rows + reasons),
--         silver.orders_clean (good rows),
--         quarantine.orders (bad rows, appended once per row)
-- =====================================================================

CREATE OR REPLACE TRANSIENT TABLE silver.orders_checked AS

WITH unified AS (

  -- The two APIs name and nest fields differently;
  -- map both to one set of columns.

  SELECT ingest_seq, source,
         raw:order_id::STRING   AS order_id,
         raw:cust_email::STRING AS cust_email,
         raw:cust_name::STRING  AS cust_name,
         raw:order_date::STRING AS order_date,
         raw:amount::STRING     AS amount,
         raw:currency::STRING   AS currency,
         raw:pincode::STRING    AS pincode,
         raw:updated_at::STRING AS updated_at

  FROM bronze.orders_raw
  WHERE source = 'web'

  UNION ALL

  SELECT ingest_seq, source,
         raw:orderId::STRING,
         raw:customer.email::STRING,
         raw:customer.name::STRING,
         raw:orderDate::STRING,
         raw:total.value::STRING,
         raw:total.currency::STRING,
         raw:deliveryPin::STRING,
         raw:updatedAt::STRING

  FROM bronze.orders_raw
  WHERE source = 'mobile'

),

typed AS (

  SELECT
    ingest_seq,
    source,

    TRIM(order_id) AS order_id,

    LOWER(TRIM(cust_email)) AS email,

    INITCAP(RTRIM(TRIM(cust_name), '.')) AS name,

    COALESCE(
      TRY_TO_DATE(order_date, 'YYYY-MM-DD'),
      TRY_TO_DATE(order_date, 'DD/MM/YYYY'),
      TRY_TO_DATE(order_date, 'MON DD, YYYY')
    ) AS order_date,

    TRY_TO_DECIMAL(
      REGEXP_REPLACE(amount, '[^0-9.-]', ''),
      12,
      2
    ) AS amount,

    UPPER(TRIM(currency)) AS currency,

    TRIM(pincode) AS pincode,

    TRY_TO_TIMESTAMP_NTZ(updated_at) AS updated_at

  FROM unified

)

SELECT *,

  ARRAY_CONSTRUCT_COMPACT(

    IFF(order_id IS NULL, 'missing_order_id', NULL),

    IFF(
    email IS NULL OR
    NOT REGEXP_LIKE(
        email,
        '^[^@\\s]+@[^@\\s]+\\.[a-z]{2,}$'
    ),
    'bad_email',
    NULL
  ),

    IFF(
      pincode IS NULL OR
      NOT REGEXP_LIKE(pincode, '^[1-9][0-9]{5}$'),
      'bad_pincode',
      NULL
    ),

    IFF(order_date IS NULL, 'unparseable_date', NULL),

    IFF(updated_at IS NULL, 'unparseable_updated_at', NULL),

    IFF(
    amount <= 0,
    'non_positive_amount',
    NULL
    )

  ) AS reasons

FROM typed;


-- =====================================================================
-- GOOD ROWS
-- =====================================================================

CREATE OR REPLACE TABLE silver.orders_clean AS

SELECT
  * EXCLUDE reasons,
  (amount IS NULL) AS amount_missing

FROM silver.orders_checked

WHERE ARRAY_SIZE(reasons) = 0;


-- =====================================================================
-- BAD ROWS: QUARANTINE
-- =====================================================================

INSERT INTO quarantine.orders
  (run_id, ingest_seq, source, order_id, reasons, row_data)

SELECT
  $run_id,
  c.ingest_seq,
  c.source,
  c.order_id,
  c.reasons,
  OBJECT_CONSTRUCT(*)

FROM silver.orders_checked c

WHERE ARRAY_SIZE(c.reasons) > 0

  AND NOT EXISTS (

        SELECT 1

        FROM quarantine.orders q

        WHERE q.source = c.source

          AND EQUAL_NULL(q.order_id, c.order_id)

          AND EQUAL_NULL(
            q.row_data:UPDATED_AT::TIMESTAMP_NTZ,
            c.updated_at
          )

  )

QUALIFY ROW_NUMBER() OVER (
  PARTITION BY c.source, c.order_id, c.updated_at
  ORDER BY c.ingest_seq DESC
) = 1;

"""

print("SQL_10_ORDERS_CLEAN successfully defined!")

SQL_10_ORDERS_CLEAN successfully defined!


In [25]:
# ================================================================
# STAGE 4.3 - CELL 2: ORDERS DEDUPLICATION
# ================================================================

# Store deduplication SQL for later execution so each order
# keeps only its newest valid version.

SQL_20_ORDERS_DEDUP = r"""

-- =====================================================================
-- SILVER 2: one row per order_id, newest version wins.
-- Ties on updated_at are broken by arrival order,
-- so reruns give the same answer.
-- =====================================================================

CREATE OR REPLACE TABLE silver.orders_dedup AS

SELECT *

FROM silver.orders_clean

QUALIFY ROW_NUMBER() OVER (

          PARTITION BY order_id

          ORDER BY updated_at DESC, ingest_seq DESC

        ) = 1;

"""

print("SQL_20_ORDERS_DEDUP successfully defined!")

SQL_20_ORDERS_DEDUP successfully defined!


In [26]:
# ================================================================
# STAGE 4.3 - CELL 3: CUSTOMER SCD TYPE 2
# ================================================================

# Store customer-history SQL for later execution so each order
# can use the customer tier valid on its date.

SQL_30_CUSTOMERS_SCD2 = r"""

-- =====================================================================
-- DIM: build the SCD Type 2 customer table from the CRM's tier-change log.
-- Each change starts a period; the period ends the day before the next change.
-- =====================================================================

CREATE OR REPLACE TABLE dim.customers_scd2 AS

WITH events AS (

    SELECT
        raw:cust_id::STRING AS cust_id,

        LOWER(TRIM(raw:email::STRING)) AS email,

        INITCAP(TRIM(raw:tier::STRING)) AS tier,

        TRY_TO_DATE(raw:effective_from::STRING) AS valid_from,

        loaded_at,
        ingest_seq

    FROM bronze.customers_raw

    -- The same event may be extracted on many runs;
    -- keep the latest copy.

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY cust_id, valid_from
        ORDER BY loaded_at DESC, ingest_seq DESC
    ) = 1

)

SELECT
    cust_id,
    email,
    tier,
    valid_from,

    COALESCE(
        DATEADD(
            day,
            -1,
            LEAD(valid_from) OVER (
                PARTITION BY cust_id
                ORDER BY valid_from
            )
        ),
        '9999-12-31'::DATE
    ) AS valid_to,

    LEAD(valid_from) OVER (
        PARTITION BY cust_id
        ORDER BY valid_from
    ) IS NULL AS is_current

FROM events

WHERE cust_id IS NOT NULL
  AND valid_from IS NOT NULL;

"""

print("SQL_30_CUSTOMERS_SCD2 successfully defined!")

SQL_30_CUSTOMERS_SCD2 successfully defined!


In [27]:
# ================================================================
# STAGE 4.3 - CELL 4: DAILY FX PROCESSING
# ================================================================

# Store daily exchange-rate SQL for later execution
# so orders can be converted to INR even on missing-rate days.

SQL_31_FX_DAILY = r"""

-- =====================================================================
-- REF: daily FX rates to INR, with no gaps.
-- 1. Flatten the API's {"rates": {"2026-09-01": {"INR": 83.2}, ...}} into rows.
-- 2. Rates are not published on weekends/holidays: fill each missing day with the
--    previous published rate, so every order date finds a rate.
-- 3. Add INR = 1.0 for every day, so rupee orders join like any other.
-- =====================================================================

CREATE OR REPLACE TABLE ref.fx_daily AS

WITH published AS (

  SELECT
      r.raw:base::STRING          AS currency,
      f.key::DATE                 AS rate_date,
      f.value:INR::NUMBER(12,4)   AS rate_to_inr

  FROM bronze.fx_raw r,
       LATERAL FLATTEN(input => r.raw:rates) f

  QUALIFY ROW_NUMBER() OVER (
      PARTITION BY currency, rate_date
      ORDER BY r.loaded_at DESC, r.ingest_seq DESC
  ) = 1

),

bounds AS (

  SELECT
      MIN(rate_date) AS d0,
      GREATEST(MAX(rate_date), CURRENT_DATE()) AS d1

  FROM published

),

spine AS (

  -- One row per calendar day

  SELECT
      DATEADD(
          day,
          ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1,
          b.d0
      ) AS d

  FROM bounds b,
       TABLE(GENERATOR(ROWCOUNT => 4000))

  QUALIFY d <= (SELECT d1 FROM bounds)

),

filled AS (

  SELECT
      c.currency,
      s.d AS rate_date,

      LAST_VALUE(p.rate_to_inr) IGNORE NULLS OVER (
          PARTITION BY c.currency
          ORDER BY s.d
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
      ) AS rate_to_inr,

      p.rate_to_inr IS NULL AS is_carried_forward

  FROM spine s

  CROSS JOIN (
      SELECT DISTINCT currency FROM published
  ) c

  LEFT JOIN published p
      ON p.currency = c.currency
     AND p.rate_date = s.d

)

SELECT
    currency,
    rate_date,
    rate_to_inr,
    is_carried_forward

FROM filled

UNION ALL

SELECT
    'INR',
    d,
    1.0,
    FALSE

FROM spine;

"""

print("SQL_31_FX_DAILY successfully defined!")

SQL_31_FX_DAILY successfully defined!


In [28]:
# ================================================================
# STAGE 4.3 - CELL 5: PINCODE GEOGRAPHY PROCESSING
# ================================================================

# Store pincode-lookup SQL for later execution so orders can
# be enriched with city, state, and sales region.

SQL_32_PINCODE_GEO = r"""

-- =====================================================================
-- REF: add newly looked-up pincodes to ref.pincode_geo.
-- India Post gives district and state; the company's sales region comes from
-- the ref.state_region seed table.
-- =====================================================================

MERGE INTO ref.pincode_geo t

USING (

    SELECT
        p.pincode,
        p.city,
        p.state,
        sr.region

    FROM (

        SELECT
            raw:pincode::STRING AS pincode,

            raw:response[0]:PostOffice[0]:District::STRING AS city,

            raw:response[0]:PostOffice[0]:State::STRING AS state,

            loaded_at,
            ingest_seq

        FROM bronze.pincode_raw

        WHERE raw:response[0]:Status::STRING = 'Success'

        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY pincode
            ORDER BY loaded_at DESC, ingest_seq DESC
        ) = 1

    ) p

    LEFT JOIN ref.state_region sr
        ON sr.state = p.state

) s

ON t.pincode = s.pincode

WHEN MATCHED AND (
    t.city <> s.city
    OR t.state <> s.state
    OR NVL(t.region, '') <> NVL(s.region, '')
)

THEN UPDATE SET
    city = s.city,
    state = s.state,
    region = s.region,
    updated_at = CURRENT_TIMESTAMP()

WHEN NOT MATCHED

THEN INSERT (
    pincode,
    city,
    state,
    region,
    updated_at
)

VALUES (
    s.pincode,
    s.city,
    s.state,
    s.region,
    CURRENT_TIMESTAMP()
);

"""

print("SQL_32_PINCODE_GEO successfully defined!")

SQL_32_PINCODE_GEO successfully defined!


In [29]:
# ================================================================
# STAGE 4.3 - CELL 6: GOLD LAYER
# DATA ENRICHMENT AND PUBLISHING
# ================================================================

# Store enrichment and publishing SQL for later execution
# so clean orders become business-ready gold records.

SQL_40_GOLD = r"""

-- =====================================================================
-- ENRICH + PUBLISH
-- 1. silver.orders_enriched: point-in-time joins to customer tier and FX rate,
--    computed columns, geography. LEFT JOINs so no order is silently lost.
-- 2. gold.fct_orders: idempotent MERGE. Re-running changes nothing unless a
--    value actually changed (row_hash).
-- =====================================================================

CREATE OR REPLACE TABLE silver.orders_enriched AS

SELECT o.*,
       c.cust_id, c.tier,
       m.multiplier,
       fx.rate_to_inr                              AS fx_rate,
       ROUND(o.amount * fx.rate_to_inr, 2)         AS amount_inr,
       FLOOR(amount_inr / 100) * m.multiplier      AS loyalty_points,
       amount_inr >= 2000                          AS is_high_value,
       g.city, g.state, g.region

FROM silver.orders_dedup o

LEFT JOIN dim.customers_scd2 c
       ON  c.email = o.email
       AND o.order_date BETWEEN c.valid_from AND c.valid_to

LEFT JOIN ref.tier_multiplier m
       ON m.tier = c.tier

LEFT JOIN ref.fx_daily fx
       ON fx.currency = o.currency
       AND fx.rate_date = o.order_date

LEFT JOIN ref.pincode_geo g
       ON g.pincode = o.pincode;


-- =====================================================================
-- PUBLISH TO GOLD
-- MERGE prevents duplicate orders when the pipeline is rerun.
-- =====================================================================

MERGE INTO gold.fct_orders t

USING (

  SELECT order_id, order_date, cust_id, email, tier, multiplier,
         amount, currency, fx_rate, amount_inr, loyalty_points,
         is_high_value, city, state, region, updated_at,

         HASH(
             order_date, cust_id, email, tier, multiplier,
             amount, currency, fx_rate, amount_inr,
             loyalty_points, is_high_value, city, state,
             region, updated_at
         ) AS row_hash

  FROM silver.orders_enriched

) s

ON t.order_id = s.order_id

WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET

  order_date = s.order_date,
  cust_id = s.cust_id,
  email = s.email,
  tier = s.tier,
  multiplier = s.multiplier,
  amount = s.amount,
  currency = s.currency,
  fx_rate = s.fx_rate,
  amount_inr = s.amount_inr,
  loyalty_points = s.loyalty_points,
  is_high_value = s.is_high_value,
  city = s.city,
  state = s.state,
  region = s.region,
  updated_at = s.updated_at,
  row_hash = s.row_hash,
  published_at = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT

  (
    order_id, order_date, cust_id, email, tier, multiplier,
    amount, currency, fx_rate, amount_inr, loyalty_points,
    is_high_value, city, state, region, updated_at,
    row_hash, published_at
  )

VALUES

  (
    s.order_id, s.order_date, s.cust_id, s.email, s.tier,
    s.multiplier, s.amount, s.currency, s.fx_rate,
    s.amount_inr, s.loyalty_points, s.is_high_value,
    s.city, s.state, s.region, s.updated_at,
    s.row_hash, CURRENT_TIMESTAMP()
  );

"""

print("SQL_40_GOLD successfully defined!")

SQL_40_GOLD successfully defined!


### Snowflake connection note

The notebook uses the single Snowflake connection established near the beginning, before SQL setup. Do not reconnect here: reconnecting would create a new session and lose the active warehouse/database context.


In [30]:
# ============================================================
# STAGE 5.1 - CELL 10: SNOWFLAKE DATA LOADING FUNCTIONS
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# FUNCTION 1: Execute SQL and retrieve results
# ------------------------------------------------------------

def query(sql):

    with conn.cursor() as cur:
        cur.execute(sql)
        return cur.fetchall()


# ------------------------------------------------------------
# FUNCTION 2: Upload files and load data into Bronze
# ------------------------------------------------------------

def load_file(local_path, table, stage_dir):

    """
    Upload one raw file to Snowflake and copy its records
    into the specified Bronze table.
    """

    if local_path is None:
        print("No file available. Skipping.")
        return 0

    # Convert local file path into Snowflake-compatible URI
    uri = "file://" + Path(local_path).resolve().as_posix()

    # STEP 1: Upload file to Snowflake internal stage

    query(
        f"""
        PUT '{uri}'
        @bronze.landing/{stage_dir}/
        AUTO_COMPRESS=FALSE
        OVERWRITE=FALSE
        """
    )

    # STEP 2: Load JSON records into Bronze table

    rows = query(f"""
        COPY INTO {table}
        (source, raw, run_id, extracted_at, src_file, src_row)

        FROM (

            SELECT
                $1:source::STRING,
                $1:data,
                $1:run_id::STRING,
                $1:extracted_at::TIMESTAMP_TZ,
                METADATA$FILENAME,
                METADATA$FILE_ROW_NUMBER

            FROM @bronze.landing/{stage_dir}/

        )

        FILE_FORMAT = (
            FORMAT_NAME = 'bronze.ff_ndjson'
        )

        ON_ERROR = 'ABORT_STATEMENT'
    """)

    # STEP 3: Calculate loaded rows

    loaded = sum(
        int(r[3])
        for r in rows
        if len(r) > 3
    )

    print(f"{loaded} rows successfully loaded into {table}")

    return loaded


print("Snowflake loading functions defined successfully!")

Snowflake loading functions defined successfully!


In [31]:
# ============================================================
# STAGE 5.1 - CELL 10: SNOWFLAKE DATA LOADING FUNCTIONS
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# FUNCTION 1: Execute SQL and retrieve results
# ------------------------------------------------------------

def query(sql):

    with conn.cursor() as cur:
        cur.execute(sql)
        return cur.fetchall()


# ------------------------------------------------------------
# FUNCTION 2: Upload files and load data into Bronze
# ------------------------------------------------------------

def load_file(local_path, table, stage_dir):

    """
    Upload one raw file to Snowflake and copy its records
    into the specified Bronze table.
    """

    if local_path is None:
        print("No file available. Skipping.")
        return 0

    # Convert local file path into Snowflake-compatible URI
    uri = "file://" + Path(local_path).resolve().as_posix()

    # STEP 1: Upload file to Snowflake internal stage

    query(
        f"""
        PUT '{uri}'
        @bronze.landing/{stage_dir}/
        AUTO_COMPRESS=FALSE
        OVERWRITE=FALSE
        """
    )

    # STEP 2: Load JSON records into Bronze table

    rows = query(f"""
        COPY INTO {table}
        (source, raw, run_id, extracted_at, src_file, src_row)

        FROM (

            SELECT
                $1:source::STRING,
                $1:data,
                $1:run_id::STRING,
                $1:extracted_at::TIMESTAMP_TZ,
                METADATA$FILENAME,
                METADATA$FILE_ROW_NUMBER

            FROM @bronze.landing/{stage_dir}/

        )

        FILE_FORMAT = (
            FORMAT_NAME = 'bronze.ff_ndjson'
        )

        ON_ERROR = 'ABORT_STATEMENT'
    """)

    # STEP 3: Calculate loaded rows

    loaded = sum(
        int(r[3])
        for r in rows
        if len(r) > 3
    )

    print(f"{loaded} rows successfully loaded into {table}")

    return loaded


print("Snowflake loading functions defined successfully!")

Snowflake loading functions defined successfully!


In [32]:
# ============================================================
# STAGE 5.2 - CELL 11: EXTRACT AND LOAD DATA INTO SNOWFLAKE
# ============================================================

from datetime import datetime, timezone
from pathlib import Path
import os

# ------------------------------------------------------------
# STEP 1: Verify required functions and sample data
# ------------------------------------------------------------

assert callable(write_landing), "write_landing() is missing"
assert callable(load_file), "load_file() is missing"

assert len(MOCK_WEB_ORDERS) == 6
assert len(MOCK_MOBILE_ORDERS) == 2
assert len(MOCK_CRM_TIER_EVENTS) == 4
assert len(MOCK_FX_RESPONSES) == 1

print("All sample datasets verified successfully!")

# ------------------------------------------------------------
# STEP 2: Set Snowflake database and warehouse
# ------------------------------------------------------------

# Use the exact warehouse and database created in SQL_00_SETUP.
# The setup cell creates the FRESHCART project database.
query("USE WAREHOUSE FRESHCART_WH")
query("USE DATABASE FRESHCART")

DATABASE_NAME = "FRESHCART"

print("Using database:", DATABASE_NAME)
print("Using warehouse: FRESHCART_WH")

# ------------------------------------------------------------
# STEP 3: Generate unique run ID
# ------------------------------------------------------------

run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

print("\nPipeline Run ID:", run_id)

# ------------------------------------------------------------
# STEP 4: Define landing directory
# ------------------------------------------------------------

LANDING_DIR = "/content/freshcart_landing"

os.makedirs(LANDING_DIR, exist_ok=True)

# ------------------------------------------------------------
# STEP 5: EXTRACT - Save sample API responses as raw files
# ------------------------------------------------------------

print("\n============================================")
print("STAGE 1: EXTRACT")
print("============================================")

web_file = write_landing(
    "web",
    MOCK_WEB_ORDERS,
    run_id
)

mobile_file = write_landing(
    "mobile",
    MOCK_MOBILE_ORDERS,
    run_id
)

crm_file = write_landing(
    "crm",
    MOCK_CRM_TIER_EVENTS,
    run_id
)

fx_file = write_landing(
    "fx",
    MOCK_FX_RESPONSES,
    run_id
)

print("\nAll four raw files generated successfully!")

# ------------------------------------------------------------
# STEP 6: LOAD - Upload files into Snowflake Bronze
# ------------------------------------------------------------

print("\n============================================")
print("STAGE 2: LOAD")
print("============================================")

web_loaded = load_file(
    web_file,
    "bronze.orders_raw",
    f"orders_web/{run_id}"
)

mobile_loaded = load_file(
    mobile_file,
    "bronze.orders_raw",
    f"orders_mobile/{run_id}"
)

crm_loaded = load_file(
    crm_file,
    "bronze.customers_raw",
    f"crm/{run_id}"
)

fx_loaded = load_file(
    fx_file,
    "bronze.fx_raw",
    f"fx/{run_id}"
)

# ------------------------------------------------------------
# STEP 7: Display loading summary
# ------------------------------------------------------------

print("\n============================================")
print("DATA LOADING SUMMARY")
print("============================================")

print("Website orders:", web_loaded)
print("Mobile orders:", mobile_loaded)
print("CRM records:", crm_loaded)
print("FX records:", fx_loaded)

total_loaded = (
    web_loaded +
    mobile_loaded +
    crm_loaded +
    fx_loaded
)

print("--------------------------------------------")
print("TOTAL RECORDS LOADED:", total_loaded)
print("============================================")

assert web_loaded == 6
assert mobile_loaded == 2
assert crm_loaded == 4
assert fx_loaded == 1

print("\nSUCCESS: All 13 records loaded into Snowflake!")


All sample datasets verified successfully!
Using database: FRESHCART
Using warehouse: FRESHCART_WH

Pipeline Run ID: 20261108_093539

STAGE 1: EXTRACT
   web: 6 records -> /content/freshcart_landing/web/web__20261108_093539.json.gz
   mobile: 2 records -> /content/freshcart_landing/mobile/mobile__20261108_093539.json.gz
   crm: 4 records -> /content/freshcart_landing/crm/crm__20261108_093539.json.gz
   fx: 1 records -> /content/freshcart_landing/fx/fx__20261108_093539.json.gz

All four raw files generated successfully!

STAGE 2: LOAD


6 rows successfully loaded into bronze.orders_raw


2 rows successfully loaded into bronze.orders_raw


4 rows successfully loaded into bronze.customers_raw


1 rows successfully loaded into bronze.fx_raw

DATA LOADING SUMMARY
Website orders: 6
Mobile orders: 2
CRM records: 4
FX records: 1
--------------------------------------------
TOTAL RECORDS LOADED: 13

SUCCESS: All 13 records loaded into Snowflake!


In [33]:
# ============================================================
# STAGE 5.2 - CELL 12: VERIFY BRONZE DATA
# ============================================================

print("WEBSITE AND MOBILE ORDERS")

orders = query(f"""
SELECT
    source,
    COUNT(*) AS total_records
FROM bronze.orders_raw
WHERE run_id = '{run_id}'
GROUP BY source
ORDER BY source
""")

for row in orders:
    print(row)


print("\nCUSTOMER RECORDS")

crm = query(f"""
SELECT COUNT(*)
FROM bronze.customers_raw
WHERE run_id = '{run_id}'
""")

print("CRM records:", crm[0][0])


print("\nFOREIGN EXCHANGE RECORDS")

fx = query(f"""
SELECT COUNT(*)
FROM bronze.fx_raw
WHERE run_id = '{run_id}'
""")

print("FX records:", fx[0][0])


print("\nTOTAL BRONZE RECORDS")

total = query(f"""
SELECT
    (SELECT COUNT(*) FROM bronze.orders_raw
     WHERE run_id = '{run_id}')
    +
    (SELECT COUNT(*) FROM bronze.customers_raw
     WHERE run_id = '{run_id}')
    +
    (SELECT COUNT(*) FROM bronze.fx_raw
     WHERE run_id = '{run_id}')
""")

print("Total:", total[0][0])

WEBSITE AND MOBILE ORDERS
('mobile', 2)
('web', 6)

CUSTOMER RECORDS
CRM records: 4

FOREIGN EXCHANGE RECORDS
FX records: 1

TOTAL BRONZE RECORDS
Total: 13


In [70]:
# ============================================================
# STAGE 6.1 - CELL 13: CLEANING AND VALIDATION
# ============================================================

SQL_10_ORDERS_CLEAN = r"""

-- STEP 1: UNIFY WEB AND MOBILE DATA

CREATE OR REPLACE TRANSIENT TABLE silver.orders_checked AS

WITH unified AS (

    SELECT
        ingest_seq,
        source,
        raw:order_id::STRING AS order_id,
        raw:cust_email::STRING AS cust_email,
        raw:cust_name::STRING AS cust_name,
        raw:order_date::STRING AS order_date,
        raw:amount::STRING AS amount,
        raw:currency::STRING AS currency,
        raw:pincode::STRING AS pincode,
        raw:updated_at::STRING AS updated_at

    FROM bronze.orders_raw
    WHERE source = 'web'

    UNION ALL

    SELECT
        ingest_seq,
        source,
        raw:orderId::STRING,
        raw:customer.email::STRING,
        raw:customer.name::STRING,
        raw:orderDate::STRING,
        raw:total.value::STRING,
        raw:total.currency::STRING,
        raw:deliveryPin::STRING,
        raw:updatedAt::STRING

    FROM bronze.orders_raw
    WHERE source = 'mobile'

),

-- STEP 2: CLEAN AND CONVERT DATA TYPES

typed AS (

    SELECT
        ingest_seq,
        source,

        TRIM(order_id) AS order_id,

        LOWER(TRIM(cust_email)) AS email,

        INITCAP(RTRIM(TRIM(cust_name), '.')) AS name,

        COALESCE(
            TRY_TO_DATE(order_date, 'YYYY-MM-DD'),
            TRY_TO_DATE(order_date, 'DD/MM/YYYY'),
            TRY_TO_DATE(order_date, 'MON DD, YYYY')
        ) AS order_date,

        TRY_TO_DECIMAL(
            REGEXP_REPLACE(amount, '[^0-9.-]', ''),
            12,
            2
        ) AS amount,

        UPPER(TRIM(currency)) AS currency,

        TRIM(pincode) AS pincode,

        TRY_TO_TIMESTAMP_NTZ(updated_at) AS updated_at

    FROM unified

)

-- STEP 3: VALIDATE EACH RECORD

SELECT *,

    ARRAY_CONSTRUCT_COMPACT(

        IFF(
            order_id IS NULL,
            'missing_order_id',
            NULL
        ),

        IFF(
            email IS NULL OR
            NOT REGEXP_LIKE(
                email,
                '^[^@\\s]+@[^@\\s]+\\.[a-z]{2,}$'
            ),
            'bad_email',
            NULL
        ),

        IFF(
            pincode IS NULL OR
            NOT REGEXP_LIKE(pincode, '^[1-9][0-9]{5}$'),
            'bad_pincode',
            NULL
        ),

        IFF(
            order_date IS NULL,
            'unparseable_date',
            NULL
        ),

        IFF(
            updated_at IS NULL,
            'unparseable_updated_at',
            NULL
        ),

        IFF(
            amount <= 0,
            'non_positive_amount',
            NULL
        ),

        IFF(
            currency NOT IN ('INR', 'USD'),
            'unsupported_currency',
            NULL
        )

    ) AS reasons

FROM typed;


-- STEP 4: SAVE VALID RECORDS INTO SILVER

CREATE OR REPLACE TABLE silver.orders_clean AS

SELECT
    * EXCLUDE reasons,
    (amount IS NULL) AS amount_missing

FROM silver.orders_checked

WHERE ARRAY_SIZE(reasons) = 0;


-- STEP 5: RECONCILE QUARANTINE WITH THE LATEST VALIDATION RESULTS

-- Remove prior quarantine entries for the same source/order/update timestamp.
-- This repairs stale false-positive entries from an earlier validation run.
DELETE FROM quarantine.orders q
USING silver.orders_checked c
WHERE q.source = c.source
  AND EQUAL_NULL(q.order_id, c.order_id)
  AND EQUAL_NULL(
      q.row_data:UPDATED_AT::TIMESTAMP_NTZ,
      c.updated_at
  );


-- STEP 6: INSERT THE CURRENT INVALID RECORDS INTO QUARANTINE

INSERT INTO quarantine.orders
(
    run_id,
    ingest_seq,
    source,
    order_id,
    reasons,
    row_data
)

SELECT
    $run_id,
    c.ingest_seq,
    c.source,
    c.order_id,
    c.reasons,
    OBJECT_CONSTRUCT(*)

FROM silver.orders_checked c

WHERE ARRAY_SIZE(c.reasons) > 0

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY
        c.source,
        c.order_id,
        c.updated_at
    ORDER BY c.ingest_seq DESC
) = 1;


"""

print("Cleaning and validation SQL defined successfully!")

Cleaning and validation SQL defined successfully!


In [35]:
# ============================================================
# STAGE 6.1 - CELL 14: EXECUTE CLEANING SQL
# ============================================================

# Set the current pipeline run ID for the quarantine INSERT.

with conn.cursor() as cur:

    cur.execute(
        f"SET run_id = '{run_id}'"
    )

# Execute all SQL statements.

results = conn.execute_string(
    SQL_10_ORDERS_CLEAN
)

print("Cleaning SQL executed successfully!")

print("Silver and quarantine tables created/updated.")

Cleaning SQL executed successfully!
Silver and quarantine tables created/updated.


In [36]:
# ============================================================
# STAGE 6.1 - CELL 15: VERIFY CLEANING RESULTS
# ============================================================

print("1. TOTAL RAW ORDERS")

raw_count = query("""
SELECT COUNT(*)
FROM bronze.orders_raw
""")

print("Raw orders:", raw_count[0][0])


print("\n2. CLEAN ORDERS")

clean_count = query("""
SELECT COUNT(*)
FROM silver.orders_clean
""")

print("Valid orders:", clean_count[0][0])


print("\n3. QUARANTINED ORDERS")

quarantine_count = query("""
SELECT COUNT(*)
FROM quarantine.orders
""")

print("Quarantined orders:", quarantine_count[0][0])


print("\n4. REJECTED ORDER DETAILS")

rejected = query("""
SELECT
    order_id,
    reasons
FROM silver.orders_checked
WHERE ARRAY_SIZE(reasons) > 0
ORDER BY order_id
""")

for row in rejected:
    print(row)


print("\n5. CLEANING SUMMARY")

print("Raw orders:", raw_count[0][0])
print("Valid orders:", clean_count[0][0])
print("Quarantined orders:", quarantine_count[0][0])

1. TOTAL RAW ORDERS
Raw orders: 8

2. CLEAN ORDERS
Valid orders: 6

3. QUARANTINED ORDERS
Quarantined orders: 2

4. REJECTED ORDER DETAILS
('ORD-1004', '[\n  "non_positive_amount"\n]')
('ORD-1006', '[\n  "bad_email",\n  "bad_pincode",\n  "unsupported_currency"\n]')

5. CLEANING SUMMARY
Raw orders: 8
Valid orders: 6
Quarantined orders: 2


In [37]:
# ============================================================
# STAGE 6.2 - CELL 16: ORDER DEDUPLICATION
# ============================================================

SQL_20_ORDERS_DEDUP = r"""

-- ============================================================
-- SILVER 2: ONE ROW PER ORDER_ID
-- Keep the newest version of every order.
-- ============================================================

CREATE OR REPLACE TABLE silver.orders_dedup AS

SELECT *

FROM silver.orders_clean

QUALIFY ROW_NUMBER() OVER (

    PARTITION BY order_id

    ORDER BY updated_at DESC, ingest_seq DESC

) = 1;

"""

print("Deduplication SQL defined successfully!")

Deduplication SQL defined successfully!


In [38]:
# ============================================================
# STAGE 6.2 - CELL 17: EXECUTE DEDUPLICATION
# ============================================================

results = conn.execute_string(SQL_20_ORDERS_DEDUP)

print("Deduplication executed successfully!")

print("Table created: SILVER.ORDERS_DEDUP")

Deduplication executed successfully!
Table created: SILVER.ORDERS_DEDUP


In [39]:
# ============================================================
# STAGE 6.2 - CELL 18: VERIFY DEDUPLICATION
# ============================================================

print("TOTAL CLEAN ORDERS")

clean = query("""
SELECT COUNT(*)
FROM silver.orders_clean
""")

print("Before deduplication:", clean[0][0])


print("\nTOTAL DEDUPLICATED ORDERS")

dedup = query("""
SELECT COUNT(*)
FROM silver.orders_dedup
""")

print("After deduplication:", dedup[0][0])


print("\nDEDUPLICATED ORDER DETAILS")

orders = query("""
SELECT
    order_id,
    email,
    amount,
    currency,
    updated_at,
    source
FROM silver.orders_dedup
ORDER BY order_id
""")

for row in orders:
    print(row)

TOTAL CLEAN ORDERS
Before deduplication: 6

TOTAL DEDUPLICATED ORDERS
After deduplication: 4

DEDUPLICATED ORDER DETAILS
('ORD-1001', 'priya.r@gmail.com', Decimal('1250.00'), 'INR', datetime.datetime(2026, 9, 1, 10, 2), 'web')
('ORD-1002', 'arjun.k@yahoo.in', Decimal('42.50'), 'USD', datetime.datetime(2026, 9, 1, 11, 15), 'mobile')
('ORD-1003', 'meera@outlook.com', Decimal('2100.00'), 'INR', datetime.datetime(2026, 9, 2, 18, 40), 'mobile')
('ORD-1005', 'priya.r@gmail.com', Decimal('899.00'), 'INR', datetime.datetime(2026, 9, 3, 16, 45), 'web')


In [40]:
# ============================================================
# STAGE 6.3 - CELL 19: CUSTOMER SCD TYPE 2
# ============================================================

# Store customer-history SQL for execution.

SQL_30_CUSTOMERS_SCD2 = r"""

-- =====================================================================
-- DIM: Build the SCD Type 2 customer table from the CRM's tier-change log.
-- Each change starts a period; the period ends the day before the next change.
-- =====================================================================

CREATE OR REPLACE TABLE dim.customers_scd2 AS

WITH events AS (

    SELECT
        raw:cust_id::STRING AS cust_id,

        LOWER(TRIM(raw:email::STRING)) AS email,

        INITCAP(TRIM(raw:tier::STRING)) AS tier,

        TRY_TO_DATE(raw:effective_from::STRING) AS valid_from,

        loaded_at,
        ingest_seq

    FROM bronze.customers_raw

    -- Keep the latest copy of the same customer event.

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY cust_id, valid_from
        ORDER BY loaded_at DESC, ingest_seq DESC
    ) = 1

)

SELECT

    cust_id,
    email,
    tier,
    valid_from,

    COALESCE(
        DATEADD(
            day,
            -1,
            LEAD(valid_from) OVER (
                PARTITION BY cust_id
                ORDER BY valid_from
            )
        ),

        '9999-12-31'::DATE

    ) AS valid_to,

    LEAD(valid_from) OVER (
        PARTITION BY cust_id
        ORDER BY valid_from
    ) IS NULL AS is_current

FROM events

WHERE cust_id IS NOT NULL
AND valid_from IS NOT NULL;

"""

print("Customer SCD Type 2 SQL defined successfully!")

Customer SCD Type 2 SQL defined successfully!


In [41]:
# ============================================================
# STAGE 6.3 - CELL 20: EXECUTE CUSTOMER SCD TYPE 2
# ============================================================

results = conn.execute_string(SQL_30_CUSTOMERS_SCD2)

print("Customer SCD Type 2 executed successfully!")

print("Table created: DIM.CUSTOMERS_SCD2")

Customer SCD Type 2 executed successfully!
Table created: DIM.CUSTOMERS_SCD2


In [42]:
# ============================================================
# STAGE 6.3 - CELL 21: VERIFY CUSTOMER HISTORY
# ============================================================

print("CUSTOMER HISTORY")

customers = query("""

SELECT
    cust_id,
    email,
    tier,
    valid_from,
    valid_to,
    is_current

FROM dim.customers_scd2

ORDER BY cust_id, valid_from

""")

for row in customers:
    print(row)


print("\nCUSTOMER COUNT")

count = query("""

SELECT COUNT(*)
FROM dim.customers_scd2

""")

print("Total customer history records:", count[0][0])


print("\nCURRENT CUSTOMER RECORDS")

current = query("""

SELECT COUNT(*)
FROM dim.customers_scd2
WHERE is_current = TRUE

""")

print("Current customer records:", current[0][0])

CUSTOMER HISTORY
('CUST-01', 'priya.r@gmail.com', 'Silver', datetime.date(2023, 4, 10), datetime.date(2026, 9, 2), False)
('CUST-01', 'priya.r@gmail.com', 'Gold', datetime.date(2026, 9, 3), datetime.date(9999, 12, 31), True)
('CUST-02', 'arjun.k@yahoo.in', 'Silver', datetime.date(2024, 1, 15), datetime.date(9999, 12, 31), True)
('CUST-03', 'meera@outlook.com', 'Platinum', datetime.date(2022, 11, 1), datetime.date(9999, 12, 31), True)

CUSTOMER COUNT
Total customer history records: 4

CURRENT CUSTOMER RECORDS
Current customer records: 3


In [43]:
# ============================================================
# STAGE 6.4 - CELL 22: DAILY FOREIGN EXCHANGE RATES
# ============================================================

SQL_31_FX_DAILY = r"""

-- =====================================================================
-- REF: daily FX rates to INR, with no gaps.
--
-- 1. Flatten the API's rates JSON into rows.
-- 2. Fill missing dates using the previous published rate.
-- 3. Add INR = 1.0 for every day.
-- =====================================================================

CREATE OR REPLACE TABLE ref.fx_daily AS

WITH published AS (

    SELECT
        r.raw:base::STRING AS currency,

        f.key::DATE AS rate_date,

        f.value:INR::NUMBER(12,4) AS rate_to_inr

    FROM bronze.fx_raw r,

    LATERAL FLATTEN(
        INPUT => r.raw:rates
    ) f

    QUALIFY ROW_NUMBER() OVER (

        PARTITION BY currency, rate_date

        ORDER BY r.loaded_at DESC, r.ingest_seq DESC

    ) = 1

),

bounds AS (

    SELECT
        MIN(rate_date) AS d0,

        GREATEST(
            MAX(rate_date),
            CURRENT_DATE()
        ) AS d1

    FROM published

),

spine AS (

    SELECT

        DATEADD(
            day,
            ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1,
            b.d0
        ) AS d

    FROM bounds b,

    TABLE(GENERATOR(ROWCOUNT => 4000))

    QUALIFY d <= (
        SELECT d1 FROM bounds
    )

),

filled AS (

    SELECT

        c.currency,

        s.d AS rate_date,

        LAST_VALUE(p.rate_to_inr) IGNORE NULLS OVER (

            PARTITION BY c.currency

            ORDER BY s.d

            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW

        ) AS rate_to_inr,

        p.rate_to_inr IS NULL AS is_carried_forward

    FROM spine s

    CROSS JOIN (
        SELECT DISTINCT currency
        FROM published
    ) c

    LEFT JOIN published p

        ON p.currency = c.currency

        AND p.rate_date = s.d

)

SELECT
    currency,
    rate_date,
    rate_to_inr,
    is_carried_forward

FROM filled

UNION ALL

SELECT
    'INR',
    d,
    1.0,
    FALSE

FROM spine;

"""

print("FX Daily SQL defined successfully!")

FX Daily SQL defined successfully!


In [44]:
# ============================================================
# STAGE 6.4 - CELL 23: EXECUTE FX TRANSFORMATION
# ============================================================

results = conn.execute_string(SQL_31_FX_DAILY)

print("FX transformation executed successfully!")

print("Table created: REF.FX_DAILY")

FX transformation executed successfully!
Table created: REF.FX_DAILY


In [45]:
# ============================================================
# STAGE 6.4 - CELL 24: VERIFY FX DATA
# ============================================================

print("TOTAL FX RECORDS")

result = query("""
SELECT COUNT(*)
FROM ref.fx_daily
""")

print("Total records:", result[0][0])


print("\nAVAILABLE CURRENCIES")

currencies = query("""
SELECT DISTINCT currency
FROM ref.fx_daily
ORDER BY currency
""")

for row in currencies:
    print(row)


print("\nSAMPLE FX RECORDS")

records = query("""
SELECT
    currency,
    rate_date,
    rate_to_inr,
    is_carried_forward

FROM ref.fx_daily

ORDER BY currency, rate_date

LIMIT 20
""")

for row in records:
    print(row)

TOTAL FX RECORDS
Total records: 138

AVAILABLE CURRENCIES
('INR',)
('USD',)

SAMPLE FX RECORDS
('INR', datetime.date(2026, 9, 1), Decimal('1.0000'), False)
('INR', datetime.date(2026, 9, 2), Decimal('1.0000'), False)
('INR', datetime.date(2026, 9, 3), Decimal('1.0000'), False)
('INR', datetime.date(2026, 9, 4), Decimal('1.0000'), False)
('INR', datetime.date(2026, 9, 5), Decimal('1.0000'), False)
('INR', datetime.date(2026, 9, 6), Decimal('1.0000'), False)
('INR', datetime.date(2026, 9, 7), Decimal('1.0000'), False)
('INR', datetime.date(2026, 9, 8), Decimal('1.0000'), False)
('INR', datetime.date(2026, 9, 9), Decimal('1.0000'), False)
('INR', datetime.date(2026, 9, 10), Decimal('1.0000'), False)
('INR', datetime.date(2026, 9, 11), Decimal('1.0000'), False)
('INR', datetime.date(2026, 9, 12), Decimal('1.0000'), False)
('INR', datetime.date(2026, 9, 13), Decimal('1.0000'), False)
('INR', datetime.date(2026, 9, 14), Decimal('1.0000'), False)
('INR', datetime.date(2026, 9, 15), Decimal('1

In [46]:
# ============================================================
# STAGE 6.5 - CELL 25: PINCODE GEOGRAPHIC ENRICHMENT
# ============================================================

SQL_32_PINCODE_GEO = r"""

-- =====================================================================
-- REF: add newly looked-up pincodes to ref.pincode_geo.
-- India Post gives district and state; the company's sales region comes
-- from the ref.state_region seed table.
-- =====================================================================

MERGE INTO ref.pincode_geo t

USING (

    SELECT
        p.pincode,
        p.city,
        p.state,
        sr.region

    FROM (

        SELECT
            raw:pincode::STRING AS pincode,

            raw:response[0]:PostOffice[0]:District::STRING AS city,

            raw:response[0]:PostOffice[0]:State::STRING AS state,

            loaded_at,
            ingest_seq

        FROM bronze.pincode_raw

        WHERE raw:response[0]:Status::STRING = 'Success'

        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY pincode
            ORDER BY loaded_at DESC, ingest_seq DESC
        ) = 1

    ) p

    LEFT JOIN ref.state_region sr
        ON sr.state = p.state

) s

ON t.pincode = s.pincode

WHEN MATCHED AND (
    t.city <> s.city
    OR t.state <> s.state
    OR NVL(t.region,'') <> NVL(s.region,'')
)

THEN UPDATE SET
    city = s.city,
    state = s.state,
    region = s.region,
    updated_at = CURRENT_TIMESTAMP()

WHEN NOT MATCHED

THEN INSERT (
    pincode,
    city,
    state,
    region,
    updated_at
)

VALUES (
    s.pincode,
    s.city,
    s.state,
    s.region,
    CURRENT_TIMESTAMP()
);

"""

print("Pincode geographic SQL defined successfully!")

Pincode geographic SQL defined successfully!


In [50]:
# ============================================================
# FRESHCART - PINCODE EXTRACTION AND BRONZE LOADING
# ============================================================

from datetime import datetime, timezone

print("=" * 65)
print("FRESHCART - PINCODE EXTRACTION AND LOADING")
print("=" * 65)


# STEP 1: Retrieve distinct pincodes from Silver

pincode_rows = query("""
    SELECT DISTINCT pincode
    FROM silver.orders_dedup
    WHERE pincode IS NOT NULL
    ORDER BY pincode
""")

pincodes = [str(row[0]).strip() for row in pincode_rows]

print("\nSTEP 1: PINCODES IDENTIFIED")

for pin in pincodes:
    print(pin)

print("Total pincodes:", len(pincodes))


# STEP 2: Mock India Post API responses

PINCODE_MASTER = {
    "600041": {
        "District": "Chennai",
        "State": "Tamil Nadu"
    },
    "560034": {
        "District": "Bengaluru",
        "State": "Karnataka"
    },
    "400050": {
        "District": "Mumbai",
        "State": "Maharashtra"
    }
}


def mock_pincode_response(pincode):

    if pincode not in PINCODE_MASTER:
        return {
            "pincode": pincode,
            "response": [{
                "Status": "Error",
                "PostOffice": []
            }]
        }

    location = PINCODE_MASTER[pincode]

    return {
        "pincode": pincode,
        "response": [{
            "Status": "Success",
            "PostOffice": [{
                "District": location["District"],
                "State": location["State"]
            }]
        }]
    }


# STEP 3: Generate API records

pincode_records = []

for pin in pincodes:
    response = mock_pincode_response(pin)
    pincode_records.append(response)

print("\nSTEP 2: API RESPONSES GENERATED")

for record in pincode_records:
    print(record)

# STEP 4: Generate a unique run ID

PINCODE_RUN_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S"
)


# STEP 5: Save API responses to local landing

pincode_file = write_landing(
    "pincode",
    pincode_records,
    PINCODE_RUN_ID
)


# STEP 6: Upload and load into Snowflake Bronze

loaded_count = load_file(
    pincode_file,
    "bronze.pincode_raw",
    f"pincode/{PINCODE_RUN_ID}"
)


# STEP 7: Validate Bronze loading

print("\nSTEP 3: BRONZE PINCODE VALIDATION")

validation = query("""
    SELECT COUNT(*)
    FROM bronze.pincode_raw
    WHERE run_id = '""" + PINCODE_RUN_ID + """'
""")

actual_count = int(validation[0][0])

print("Expected records:", len(pincode_records))
print("Loaded records:", actual_count)

if actual_count == len(pincode_records):
    print("\nPINCODE BRONZE LOADING SUCCESSFUL!")
else:
    raise RuntimeError(
        "Pincode loading count mismatch. "
        "Expected {}, found {}.".format(
            len(pincode_records), actual_count
        )
    )

print("=" * 65)

FRESHCART - PINCODE EXTRACTION AND LOADING

STEP 1: PINCODES IDENTIFIED
400050
560034
600041
Total pincodes: 3

STEP 2: API RESPONSES GENERATED
{'pincode': '400050', 'response': [{'Status': 'Success', 'PostOffice': [{'District': 'Mumbai', 'State': 'Maharashtra'}]}]}
{'pincode': '560034', 'response': [{'Status': 'Success', 'PostOffice': [{'District': 'Bengaluru', 'State': 'Karnataka'}]}]}
{'pincode': '600041', 'response': [{'Status': 'Success', 'PostOffice': [{'District': 'Chennai', 'State': 'Tamil Nadu'}]}]}
   pincode: 3 records -> /content/freshcart_landing/pincode/pincode__20261108_095829.json.gz


3 rows successfully loaded into bronze.pincode_raw

STEP 3: BRONZE PINCODE VALIDATION
Expected records: 3
Loaded records: 3

PINCODE BRONZE LOADING SUCCESSFUL!


In [51]:
# ============================================================
# STAGE 6.5 - CELL 26: EXECUTE PINCODE TRANSFORMATION
# ============================================================

results = conn.execute_string(SQL_32_PINCODE_GEO)

print("Pincode transformation executed successfully!")

print("Table populated: REF.PINCODE_GEO")

Pincode transformation executed successfully!
Table populated: REF.PINCODE_GEO


In [52]:
# ============================================================
# STAGE 6.5 - CELL 27: VERIFY PINCODE DATA
# ============================================================

print("TOTAL PINCODES")

result = query("""
SELECT COUNT(*)
FROM ref.pincode_geo
""")

print("Total pincodes:", result[0][0])


print("\nPINCODE DETAILS")

records = query("""
SELECT
    pincode,
    city,
    state,
    region,
    updated_at

FROM ref.pincode_geo

ORDER BY pincode
""")

for row in records:
    print(row)


print("\nREGION-WISE PINCODE DISTRIBUTION")

regions = query("""
SELECT
    region,
    COUNT(*) AS total_pincodes

FROM ref.pincode_geo

GROUP BY region

ORDER BY region
""")

for row in regions:
    print(row)

TOTAL PINCODES
Total pincodes: 3

PINCODE DETAILS
('400050', 'Mumbai', 'Maharashtra', 'West', datetime.datetime(2026, 11, 8, 2, 0, 34, 828000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
('560034', 'Bengaluru', 'Karnataka', 'South', datetime.datetime(2026, 11, 8, 2, 0, 34, 828000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
('600041', 'Chennai', 'Tamil Nadu', 'South', datetime.datetime(2026, 11, 8, 2, 0, 34, 828000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))

REGION-WISE PINCODE DISTRIBUTION
('South', 2)
('West', 1)


In [53]:
# ============================================================
# FRESHCART - STAGE 6.6
# GOLD LAYER: ENRICHMENT AND PUBLISHING
# ============================================================

print("=" * 65)
print("FRESHCART - GOLD LAYER TRANSFORMATION")
print("=" * 65)

# Execute the original professor-provided SQL definition
run_sql(SQL_40_GOLD)

print("\nGold layer transformation completed successfully!")

FRESHCART - GOLD LAYER TRANSFORMATION

Gold layer transformation completed successfully!


In [54]:
gold_summary = query("""
    SELECT
        COUNT(*) AS total_orders,
        SUM(amount_inr) AS total_revenue_inr,
        SUM(loyalty_points) AS total_loyalty_points
    FROM gold.fct_orders
""")

print("GOLD BASELINE SUMMARY")
print("-" * 40)
print("Total orders:", gold_summary[0][0])
print("Total revenue:", gold_summary[0][1])
print("Total loyalty points:", gold_summary[0][2])

GOLD BASELINE SUMMARY
----------------------------------------
Total orders: 4
Total revenue: 7785.00
Total loyalty points: 126


In [55]:
# ============================================================
# FRESHCART - STAGE 6.7
# DEFINE AND EXECUTE FINAL DATA QUALITY CHECKS
# ============================================================

def quality_checks():

    print("=" * 75)
    print("FRESHCART - FINAL DATA QUALITY VALIDATION")
    print("=" * 75)

    checks = query("""

        SELECT
            '01. ROW COUNT PRESERVATION' AS check_name,

            ABS(
                (SELECT COUNT(*) FROM silver.orders_dedup)
                -
                (SELECT COUNT(*) FROM silver.orders_enriched)
            ) AS failed_rows

        UNION ALL

        SELECT
            '02. DUPLICATE ORDERS IN GOLD',

            COUNT(*) - COUNT(DISTINCT order_id)

        FROM gold.fct_orders

        UNION ALL

        SELECT
            '03. ORDERS WITHOUT MATCHING CUSTOMER',

            COUNT(*)

        FROM silver.orders_enriched

        WHERE cust_id IS NULL

        UNION ALL

        SELECT
            '04. ORDERS WITHOUT EXCHANGE RATE',

            COUNT(*)

        FROM silver.orders_enriched

        WHERE fx_rate IS NULL

        UNION ALL

        SELECT
            '05. ORDERS WITHOUT CITY INFORMATION',

            COUNT(*)

        FROM silver.orders_enriched

        WHERE city IS NULL OR TRIM(city) = ''

        UNION ALL

        SELECT
            '06. NEGATIVE LOYALTY POINTS IN GOLD',

            COUNT(*)

        FROM gold.fct_orders

        WHERE loyalty_points < 0

    """)

    failed_checks = 0

    for row in checks:

        check_name = row[0]
        failed_rows = int(row[1])

        if failed_rows == 0:
            status = "PASSED"
        else:
            status = "FAILED"
            failed_checks += 1

        print(
            f"{check_name:<48} "
            f"Failed rows: {failed_rows:<5} "
            f"{status}"
        )

    print("\n" + "=" * 75)

    if failed_checks == 0:

        print("ALL SIX DATA QUALITY CHECKS PASSED!")

    else:

        print(f"WARNING: {failed_checks} QUALITY CHECK(S) FAILED.")

        raise AssertionError(
            "FreshCart data quality validation failed."
        )

    print("=" * 75)

    return checks


# Execute validation

quality_checks()

FRESHCART - FINAL DATA QUALITY VALIDATION
01. ROW COUNT PRESERVATION                       Failed rows: 0     PASSED
02. DUPLICATE ORDERS IN GOLD                     Failed rows: 0     PASSED
03. ORDERS WITHOUT MATCHING CUSTOMER             Failed rows: 0     PASSED
04. ORDERS WITHOUT EXCHANGE RATE                 Failed rows: 0     PASSED
05. ORDERS WITHOUT CITY INFORMATION              Failed rows: 0     PASSED
06. NEGATIVE LOYALTY POINTS IN GOLD              Failed rows: 0     PASSED

ALL SIX DATA QUALITY CHECKS PASSED!


[('01. ROW COUNT PRESERVATION', 0),
 ('02. DUPLICATE ORDERS IN GOLD', 0),
 ('03. ORDERS WITHOUT MATCHING CUSTOMER', 0),
 ('04. ORDERS WITHOUT EXCHANGE RATE', 0),
 ('05. ORDERS WITHOUT CITY INFORMATION', 0),
 ('06. NEGATIVE LOYALTY POINTS IN GOLD', 0)]

In [56]:
# ============================================================
# FRESHCART - STAGE 6.8
# FINAL BUSINESS METRICS VALIDATION
# ============================================================

print("=" * 75)
print("FRESHCART - FINAL BUSINESS METRICS")
print("=" * 75)

# Retrieve all important pipeline metrics directly from Snowflake

metrics = query("""

SELECT

    (SELECT COUNT(*)
     FROM bronze.orders_raw) AS bronze_orders,

    (SELECT COUNT(*)
     FROM silver.orders_dedup) AS silver_orders,

    (SELECT COUNT(*)
     FROM silver.orders_enriched) AS enriched_orders,

    (SELECT COUNT(*)
     FROM gold.fct_orders) AS gold_orders,

    (SELECT COUNT(*)
     FROM quarantine.orders) AS quarantined_orders,

    (SELECT COALESCE(SUM(amount_inr), 0)
     FROM gold.fct_orders) AS total_revenue_inr,

    (SELECT COALESCE(SUM(loyalty_points), 0)
     FROM gold.fct_orders) AS total_loyalty_points,

    (SELECT COUNT(*) - COUNT(DISTINCT order_id)
     FROM gold.fct_orders) AS duplicate_orders

""")

result = metrics[0]

# Display results

print("\n1. PIPELINE RECORD COUNTS")
print("-" * 50)

print("Bronze Orders:       ", result[0])
print("Silver Orders:       ", result[1])
print("Enriched Orders:     ", result[2])
print("Gold Orders:         ", result[3])
print("Quarantined Orders:  ", result[4])

print("\n2. BUSINESS METRICS")
print("-" * 50)

print(f"Total Revenue:       ₹{float(result[5]):,.2f}")
print(f"Total Loyalty Points: {result[6]}")

print("\n3. DUPLICATE VALIDATION")
print("-" * 50)

print("Duplicate Gold Orders:", result[7])

# Calculate quarantine percentage against Bronze input

if int(result[0]) > 0:

    rejection_rate = (
        int(result[4]) / int(result[0])
    ) * 100

    print(f"Quarantine Rate:      {rejection_rate:.2f}%")

print("\n" + "=" * 75)
print("FINAL BUSINESS METRICS VALIDATION COMPLETED")
print("=" * 75)

FRESHCART - FINAL BUSINESS METRICS

1. PIPELINE RECORD COUNTS
--------------------------------------------------
Bronze Orders:        8
Silver Orders:        4
Enriched Orders:      4
Gold Orders:          4
Quarantined Orders:   2

2. BUSINESS METRICS
--------------------------------------------------
Total Revenue:       ₹7,785.00
Total Loyalty Points: 126

3. DUPLICATE VALIDATION
--------------------------------------------------
Duplicate Gold Orders: 0
Quarantine Rate:      25.00%

FINAL BUSINESS METRICS VALIDATION COMPLETED


# **B2**

Step 1 — Inspect the two raw ORD-1003 records

In [58]:
query("""
    SELECT
        source,
        raw,
        loaded_at,
        ingest_seq
    FROM bronze.orders_raw
    WHERE raw:order_id::STRING = 'ORD-1003'
       OR raw:orderId::STRING = 'ORD-1003'
    ORDER BY loaded_at, ingest_seq
""")

[('web',
  '{\n  "amount": null,\n  "currency": "INR",\n  "cust_email": "meera@outlook.com",\n  "cust_name": "MEERA S",\n  "order_date": "2026-09-02",\n  "order_id": "ORD-1003",\n  "pincode": "400050",\n  "updated_at": "2026-09-02 09:10:00"\n}',
  datetime.datetime(2026, 11, 8, 1, 35, 41, 367000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>),
  2),
 ('mobile',
  '{\n  "customer": {\n    "email": "Meera@Outlook.com",\n    "name": "Meera S"\n  },\n  "deliveryPin": "400050",\n  "orderDate": "02/09/2026",\n  "orderId": "ORD-1003",\n  "total": {\n    "currency": "INR",\n    "value": "2,100.00"\n  },\n  "updatedAt": "2026-09-02T18:40:00"\n}',
  datetime.datetime(2026, 11, 8, 1, 35, 43, 769000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>),
  8)]

The order `ORD-1003` appears twice in `bronze.orders_raw`, originating from two different source systems: **web** and **mobile**.

- **Web record:** Uses flat JSON fields such as `order_id`, `cust_email`, `cust_name`, `order_date`, and `amount`. Its amount is `NULL`, and its `updated_at` value is `2026-09-02 09:10:00`.
- **Mobile record:** Uses a different JSON structure. It stores the order ID as `orderId`, customer details inside a nested `customer` object (`customer.email` and `customer.name`), and the amount inside a nested `total` object (`total.value`). Its amount is `2,100.00 INR`, and its `updatedAt` value is `2026-09-02T18:40:00`.

The field names differ because the web and mobile applications send data using different API payload formats. The web application uses flat, snake_case field names, whereas the mobile application uses camelCase names and nested JSON objects. Therefore, the ELT pipeline must standardize these different structures before processing them in the Silver layer.

Step 2: Inspect the standardized records

In [59]:
query("""
    SELECT *
    FROM silver.orders_checked
    WHERE order_id = 'ORD-1003'
    ORDER BY updated_at, ingest_seq
""")

[(2,
  'web',
  'ORD-1003',
  'meera@outlook.com',
  'Meera S',
  datetime.date(2026, 9, 2),
  None,
  'INR',
  '400050',
  datetime.datetime(2026, 9, 2, 9, 10),
  '[]'),
 (8,
  'mobile',
  'ORD-1003',
  'meera@outlook.com',
  'Meera S',
  datetime.date(2026, 9, 2),
  Decimal('2100.00'),
  'INR',
  '400050',
  datetime.datetime(2026, 9, 2, 18, 40),
  '[]')]

The SQL_10_ORDERS_CLEAN transformation uses a CTE named unified with UNION ALL to map the web and mobile API fields into a common set of columns in silver.orders_checked.

Step 3: Verify which record survives deduplication

In [60]:
query("""
    SELECT *
    FROM silver.orders_dedup
    WHERE order_id = 'ORD-1003'
""")

[(8,
  'mobile',
  'ORD-1003',
  'meera@outlook.com',
  'Meera S',
  datetime.date(2026, 9, 2),
  Decimal('2100.00'),
  'INR',
  '400050',
  datetime.datetime(2026, 9, 2, 18, 40),
  False)]

The mobile record survives the deduplication process because it has the latest updated_at timestamp (2026-09-02 18:40:00), compared with the web record (2026-09-02 09:10:00).
PARTITION BY order_id groups records belonging to the same order. ORDER BY updated_at DESC ranks the latest record first, while ingest_seq DESC acts as a tie-breaker. QUALIFY ROW_NUMBER() = 1 retains only the highest-ranked record.

Therefore, the mobile record (ingest_seq = 8) is retained with an amount of ₹2,100.00, while the older web record (ingest_seq = 2) is discarded.

Step 4: Calculation of amount_inr and loyalty_points.

In [62]:
query("""
    SELECT
        order_id,
        order_date,
        tier,
        multiplier,
        amount,
        currency,
        fx_rate,
        amount_inr,
        loyalty_points
    FROM gold.fct_orders
    WHERE order_id ='ORD-1003'
    ORDER BY order_id
""")

[('ORD-1003',
  datetime.date(2026, 9, 2),
  'Platinum',
  3,
  Decimal('2100.00'),
  'INR',
  Decimal('1.0000'),
  Decimal('2100.00'),
  63)]

For ORD-1003, the Gold layer calculates `amount_inr` by multiplying the original order amount by the applicable exchange rate.

The original amount is ₹2,100.00, and since the order currency is INR, the exchange rate is 1.0000.

**amount_inr = ROUND(2100.00 × 1.0000, 2) = ₹2,100.00**

The Gold layer calculates loyalty points by dividing the INR amount by 100, applying the FLOOR function, and multiplying the result by the customer's loyalty-tier multiplier.

The customer belongs to the Platinum tier, which has a multiplier of 3.

**loyalty_points = FLOOR(2100 / 100) × 3 = 21 × 3 = 63 points.**

Therefore, ORD-1003 has a Gold-layer amount of ₹2,100.00 and earns 63 loyalty points.

Do the same, more briefly, for ORD-1005: explain why Priya earns points at the Gold multiplier for this
order but at Silver for ORD-1001. Name the table and the join condition responsible.

In [63]:
query("""
    SELECT
        o.order_id,
        o.order_date,
        o.email,
        c.tier,
        c.valid_from,
        c.valid_to,
        m.multiplier,
        o.amount,
        g.loyalty_points
    FROM silver.orders_dedup o
    JOIN dim.customers_scd2 c
        ON c.email = o.email
       AND o.order_date BETWEEN c.valid_from AND c.valid_to
    JOIN ref.tier_multiplier m
        ON m.tier = c.tier
    JOIN gold.fct_orders g
        ON g.order_id = o.order_id
    WHERE o.email = 'priya.r@gmail.com'
      AND o.order_id IN ('ORD-1001', 'ORD-1005')
    ORDER BY o.order_date;
""")

[('ORD-1001',
  datetime.date(2026, 9, 1),
  'priya.r@gmail.com',
  'Silver',
  datetime.date(2023, 4, 10),
  datetime.date(2026, 9, 2),
  1,
  Decimal('1250.00'),
  12),
 ('ORD-1005',
  datetime.date(2026, 9, 3),
  'priya.r@gmail.com',
  'Gold',
  datetime.date(2026, 9, 3),
  datetime.date(9999, 12, 31),
  2,
  Decimal('899.00'),
  16)]

For ORD-1005, Priya earns loyalty points using the Gold multiplier (2), whereas for ORD-1001, she earns points using the Silver multiplier (1).

This is handled by the `DIM.CUSTOMERS_SCD2` table, which maintains the historical records of customer loyalty tiers.

The Gold transformation uses the following join condition:

```sql
LEFT JOIN dim.customers_scd2 c
    ON c.email = o.email
   AND o.order_date BETWEEN c.valid_from AND c.valid_to
```

The email condition identifies the customer, while the date-range condition selects the tier that was valid when the order was placed.

Priya's Silver tier was valid until 2 September 2026, and her Gold tier became valid from 3 September 2026. Consequently, ORD-1001 (1 September) receives the Silver multiplier of 1, while ORD-1005 (3 September) receives the Gold multiplier of 2.

This demonstrates how SCD Type 2 preserves customer history and ensures that loyalty points are calculated using the correct tier applicable on each order date.

# **B3**

In [69]:
query("""
    SELECT
        order_id,
        currency,
        reasons
    FROM silver.orders_checked
    WHERE order_id = 'ORD-1006';
""")

[('ORD-1006', 'EUR', '[\n  "bad_email",\n  "bad_pincode"\n]')]

In [75]:
# ============================================================
# FRESHCART - STAGE 7.0
# CENTRALIZED PIPELINE ORCHESTRATION
# ============================================================

print("=" * 75)
print("FRESHCART - CENTRALIZED PIPELINE ORCHESTRATION")
print("=" * 75)


# ------------------------------------------------------------
# STEP 1: DEFINE THE TRANSFORMATION ORCHESTRATOR
# ------------------------------------------------------------

def run_freshcart_transformations():

    print("\nSTAGE 1: ORDER CLEANING AND VALIDATION")
    print("-" * 60)

    with conn.cursor() as cur:
        cur.execute(f"SET run_id = '{run_id}'")

    run_sql(SQL_10_ORDERS_CLEAN)

    print("Order cleaning completed!")


    # --------------------------------------------------------
    # STEP 2: ORDER DEDUPLICATION
    # --------------------------------------------------------

    print("\nSTAGE 2: ORDER DEDUPLICATION")
    print("-" * 60)

    run_sql(SQL_20_ORDERS_DEDUP)

    print("Order deduplication completed!")


    # --------------------------------------------------------
    # STEP 3: CUSTOMER SCD TYPE 2
    # --------------------------------------------------------

    print("\nSTAGE 3: CUSTOMER HISTORY")
    print("-" * 60)

    run_sql(SQL_30_CUSTOMERS_SCD2)

    print("Customer SCD Type 2 completed!")


    # --------------------------------------------------------
    # STEP 4: DAILY FOREIGN EXCHANGE PROCESSING
    # --------------------------------------------------------

    print("\nSTAGE 4: FOREIGN EXCHANGE PROCESSING")
    print("-" * 60)

    run_sql(SQL_31_FX_DAILY)

    print("FX processing completed!")


    # --------------------------------------------------------
    # STEP 5: PINCODE GEOGRAPHIC ENRICHMENT
    # --------------------------------------------------------

    print("\nSTAGE 5: PINCODE GEOGRAPHIC ENRICHMENT")
    print("-" * 60)

    run_sql(SQL_32_PINCODE_GEO)

    print("Pincode enrichment completed!")


    # --------------------------------------------------------
    # STEP 6: GOLD LAYER TRANSFORMATION
    # --------------------------------------------------------

    print("\nSTAGE 6: GOLD LAYER PUBLISHING")
    print("-" * 60)

    run_sql(SQL_40_GOLD)

    print("Gold layer publishing completed!")


    # --------------------------------------------------------
    # STEP 7: FINAL DATA QUALITY VALIDATION
    # --------------------------------------------------------

    print("\nSTAGE 7: DATA QUALITY VALIDATION")
    print("-" * 60)

    quality_results = quality_checks()


    # --------------------------------------------------------
    # STEP 8: FINAL BUSINESS METRICS
    # --------------------------------------------------------

    print("\nSTAGE 8: BUSINESS METRICS")
    print("-" * 60)

    metrics = query("""

        SELECT

            (SELECT COUNT(*)
             FROM bronze.orders_raw),

            (SELECT COUNT(*)
             FROM silver.orders_dedup),

            (SELECT COUNT(*)
             FROM gold.fct_orders),

            (SELECT COALESCE(SUM(amount_inr), 0)
             FROM gold.fct_orders),

            (SELECT COALESCE(SUM(loyalty_points), 0)
             FROM gold.fct_orders)

    """)

    result = metrics[0]

    print("Bronze Orders:", result[0])
    print("Silver Orders:", result[1])
    print("Gold Orders:", result[2])

    print(f"Total Revenue: ₹{float(result[3]):,.2f}")

    print("Total Loyalty Points:", result[4])


    # --------------------------------------------------------
    # STEP 9: FINAL STATUS
    # --------------------------------------------------------

    print("\n" + "=" * 75)
    print("FRESHCART TRANSFORMATION PIPELINE COMPLETED")
    print("=" * 75)

    return {
        "bronze_orders": result[0],
        "silver_orders": result[1],
        "gold_orders": result[2],
        "revenue": float(result[3]),
        "loyalty_points": result[4]
    }


# ------------------------------------------------------------
# STEP 10: EXECUTE PIPELINE
# ------------------------------------------------------------

pipeline_results = run_freshcart_transformations()

print("\nPipeline execution results:")
print(pipeline_results)

FRESHCART - CENTRALIZED PIPELINE ORCHESTRATION

STAGE 1: ORDER CLEANING AND VALIDATION
------------------------------------------------------------
Order cleaning completed!

STAGE 2: ORDER DEDUPLICATION
------------------------------------------------------------
Order deduplication completed!

STAGE 3: CUSTOMER HISTORY
------------------------------------------------------------
Customer SCD Type 2 completed!

STAGE 4: FOREIGN EXCHANGE PROCESSING
------------------------------------------------------------
FX processing completed!

STAGE 5: PINCODE GEOGRAPHIC ENRICHMENT
------------------------------------------------------------
Pincode enrichment completed!

STAGE 6: GOLD LAYER PUBLISHING
------------------------------------------------------------
Gold layer publishing completed!

STAGE 7: DATA QUALITY VALIDATION
------------------------------------------------------------
FRESHCART - FINAL DATA QUALITY VALIDATION
01. ROW COUNT PRESERVATION                       Failed rows: 0    

In [72]:
query("""
    SELECT
        order_id,
        currency,
        reasons
    FROM silver.orders_checked
    WHERE order_id = 'ORD-1006';
""")

[('ORD-1006',
  'EUR',
  '[\n  "bad_email",\n  "bad_pincode",\n  "unsupported_currency"\n]')]

The Gold totals do not change after introducing the `unsupported_currency` validation rule.

ORD-1006 was already rejected because it contained an invalid email (`bad_email`) and an invalid pincode (`bad_pincode`). Therefore, it was already excluded from the Silver and Gold layers before the new validation rule was introduced.

The new validation rule identifies EUR as an unsupported currency and adds `unsupported_currency` to the reasons array. However, it does not change the order's rejection status because the record was already invalid.

After rerunning the pipeline, the Gold layer still contains 4 orders, with total revenue of ₹7,785.00 and 126 loyalty points. All six data quality checks passed.

**ORD-1006 is already in quarantine from the earlier run. Does your new run add a second quarantine
row for it? Point to the part of the SQL that decides this.**

No, rerunning the pipeline does not create a second quarantine record for ORD-1006.

This is controlled by Step 5 and Step 6 of `SQL_10_ORDERS_CLEAN`.

In Step 5, the `DELETE FROM quarantine.orders` statement removes any previously quarantined record matching the same source, order ID and update timestamp. This ensures that an old quarantine entry for the same event is removed before inserting the latest validation result.

DELETE FROM quarantine.orders q
USING silver.orders_checked c
WHERE q.source = c.source
  AND EQUAL_NULL(q.order_id, c.order_id)
  AND EQUAL_NULL(
      q.row_data:UPDATED_AT::TIMESTAMP_NTZ,
      c.updated_at
  );

In Step 6, the `INSERT` statement inserts the currently invalid records into the quarantine table. The `QUALIFY ROW_NUMBER()` clause partitions records by `source`, `order_id` and `updated_at`, and retains only the record with the highest `ingest_seq`.

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY
        c.source,
        c.order_id,
        c.updated_at
    ORDER BY c.ingest_seq DESC
) = 1;

Therefore, the pipeline replaces the previous quarantine entry with the latest validation result instead of creating a duplicate record for the same event.

# B4

In [82]:
gold_before_b4 = query("""
    SELECT
        order_id,
        published_at
    FROM gold.fct_orders
    ORDER BY order_id;
""")

print("Gold timestamps before B4:")
for row in gold_before_b4:
    print(row)

Gold timestamps before B4:
('ORD-1001', datetime.datetime(2026, 11, 8, 2, 6, 43, 298000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
('ORD-1002', datetime.datetime(2026, 11, 8, 2, 6, 43, 298000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
('ORD-1003', datetime.datetime(2026, 11, 8, 2, 6, 43, 298000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
('ORD-1005', datetime.datetime(2026, 11, 8, 2, 6, 43, 298000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))


In [85]:
from datetime import datetime, timezone

run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")

print("New CRM Run ID:", run_id)

crm_file = write_landing(
    "crm",
    MOCK_CRM_TIER_EVENTS,
    run_id
)

crm_loaded = load_file(
    crm_file,
    "bronze.customers_raw",
    f"crm/{run_id}"
)

print("CRM records loaded:", crm_loaded)

New CRM Run ID: 20261108_162056_968890
   crm: 5 records -> /content/freshcart_landing/crm/crm__20261108_162056_968890.json.gz


5 rows successfully loaded into bronze.customers_raw
CRM records loaded: 5


In [86]:
query("""
    SELECT
        raw:cust_id::STRING AS cust_id,
        raw:tier::STRING AS tier,
        raw:effective_from::DATE AS effective_from
    FROM bronze.customers_raw
    WHERE raw:cust_id::STRING = 'CUST-02'
    ORDER BY effective_from;
""")

[('CUST-02', 'Silver', datetime.date(2024, 1, 15)),
 ('CUST-02', 'Silver', datetime.date(2024, 1, 15)),
 ('CUST-02', 'Gold', datetime.date(2026, 8, 15))]

In [87]:
# ============================================================
# FRESHCART - STAGE 7.0
# CENTRALIZED PIPELINE ORCHESTRATION
# ============================================================

print("=" * 75)
print("FRESHCART - CENTRALIZED PIPELINE ORCHESTRATION")
print("=" * 75)


# ------------------------------------------------------------
# STEP 1: DEFINE THE TRANSFORMATION ORCHESTRATOR
# ------------------------------------------------------------

def run_freshcart_transformations():

    print("\nSTAGE 1: ORDER CLEANING AND VALIDATION")
    print("-" * 60)

    with conn.cursor() as cur:
        cur.execute(f"SET run_id = '{run_id}'")

    run_sql(SQL_10_ORDERS_CLEAN)

    print("Order cleaning completed!")


    # --------------------------------------------------------
    # STEP 2: ORDER DEDUPLICATION
    # --------------------------------------------------------

    print("\nSTAGE 2: ORDER DEDUPLICATION")
    print("-" * 60)

    run_sql(SQL_20_ORDERS_DEDUP)

    print("Order deduplication completed!")


    # --------------------------------------------------------
    # STEP 3: CUSTOMER SCD TYPE 2
    # --------------------------------------------------------

    print("\nSTAGE 3: CUSTOMER HISTORY")
    print("-" * 60)

    run_sql(SQL_30_CUSTOMERS_SCD2)

    print("Customer SCD Type 2 completed!")


    # --------------------------------------------------------
    # STEP 4: DAILY FOREIGN EXCHANGE PROCESSING
    # --------------------------------------------------------

    print("\nSTAGE 4: FOREIGN EXCHANGE PROCESSING")
    print("-" * 60)

    run_sql(SQL_31_FX_DAILY)

    print("FX processing completed!")


    # --------------------------------------------------------
    # STEP 5: PINCODE GEOGRAPHIC ENRICHMENT
    # --------------------------------------------------------

    print("\nSTAGE 5: PINCODE GEOGRAPHIC ENRICHMENT")
    print("-" * 60)

    run_sql(SQL_32_PINCODE_GEO)

    print("Pincode enrichment completed!")


    # --------------------------------------------------------
    # STEP 6: GOLD LAYER TRANSFORMATION
    # --------------------------------------------------------

    print("\nSTAGE 6: GOLD LAYER PUBLISHING")
    print("-" * 60)

    run_sql(SQL_40_GOLD)

    print("Gold layer publishing completed!")


    # --------------------------------------------------------
    # STEP 7: FINAL DATA QUALITY VALIDATION
    # --------------------------------------------------------

    print("\nSTAGE 7: DATA QUALITY VALIDATION")
    print("-" * 60)

    quality_results = quality_checks()


    # --------------------------------------------------------
    # STEP 8: FINAL BUSINESS METRICS
    # --------------------------------------------------------

    print("\nSTAGE 8: BUSINESS METRICS")
    print("-" * 60)

    metrics = query("""

        SELECT

            (SELECT COUNT(*)
             FROM bronze.orders_raw),

            (SELECT COUNT(*)
             FROM silver.orders_dedup),

            (SELECT COUNT(*)
             FROM gold.fct_orders),

            (SELECT COALESCE(SUM(amount_inr), 0)
             FROM gold.fct_orders),

            (SELECT COALESCE(SUM(loyalty_points), 0)
             FROM gold.fct_orders)

    """)

    result = metrics[0]

    print("Bronze Orders:", result[0])
    print("Silver Orders:", result[1])
    print("Gold Orders:", result[2])

    print(f"Total Revenue: ₹{float(result[3]):,.2f}")

    print("Total Loyalty Points:", result[4])


    # --------------------------------------------------------
    # STEP 9: FINAL STATUS
    # --------------------------------------------------------

    print("\n" + "=" * 75)
    print("FRESHCART TRANSFORMATION PIPELINE COMPLETED")
    print("=" * 75)

    return {
        "bronze_orders": result[0],
        "silver_orders": result[1],
        "gold_orders": result[2],
        "revenue": float(result[3]),
        "loyalty_points": result[4]
    }


# ------------------------------------------------------------
# STEP 10: EXECUTE PIPELINE
# ------------------------------------------------------------

pipeline_results = run_freshcart_transformations()

print("\nPipeline execution results:")
print(pipeline_results)

FRESHCART - CENTRALIZED PIPELINE ORCHESTRATION

STAGE 1: ORDER CLEANING AND VALIDATION
------------------------------------------------------------
Order cleaning completed!

STAGE 2: ORDER DEDUPLICATION
------------------------------------------------------------
Order deduplication completed!

STAGE 3: CUSTOMER HISTORY
------------------------------------------------------------
Customer SCD Type 2 completed!

STAGE 4: FOREIGN EXCHANGE PROCESSING
------------------------------------------------------------
FX processing completed!

STAGE 5: PINCODE GEOGRAPHIC ENRICHMENT
------------------------------------------------------------
Pincode enrichment completed!

STAGE 6: GOLD LAYER PUBLISHING
------------------------------------------------------------
Gold layer publishing completed!

STAGE 7: DATA QUALITY VALIDATION
------------------------------------------------------------
FRESHCART - FINAL DATA QUALITY VALIDATION
01. ROW COUNT PRESERVATION                       Failed rows: 0    

In [88]:
gold_after_b4 = query("""
    SELECT
        order_id,
        published_at
    FROM gold.fct_orders
    ORDER BY order_id;
""")

print("Gold timestamps after B4:")
for row in gold_after_b4:
    print(row)

Gold timestamps after B4:
('ORD-1001', datetime.datetime(2026, 11, 8, 2, 6, 43, 298000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
('ORD-1002', datetime.datetime(2026, 11, 8, 8, 23, 1, 429000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
('ORD-1003', datetime.datetime(2026, 11, 8, 2, 6, 43, 298000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))
('ORD-1005', datetime.datetime(2026, 11, 8, 2, 6, 43, 298000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>))


In [89]:
before_dict = dict(gold_before_b4)
after_dict = dict(gold_after_b4)

changed_rows = sum(
    1 for order_id in before_dict
    if before_dict[order_id] != after_dict.get(order_id)
)

print("Gold rows whose published_at changed:", changed_rows)

Gold rows whose published_at changed: 1


In [91]:
# ============================================================
# FRESHCART - STAGE 7.1
# GOLD LAYER IDEMPOTENCY TESTING
# ============================================================

print("=" * 75)
print("FRESHCART - IDEMPOTENCY TESTING")
print("=" * 75)


# STEP 1: Capture Gold metrics before re-execution

def get_gold_snapshot():

    result = query("""

        SELECT
            COUNT(*) AS total_orders,
            COUNT(DISTINCT order_id) AS unique_orders,
            COALESCE(SUM(amount_inr), 0) AS total_revenue,
            COALESCE(SUM(loyalty_points), 0) AS total_points

        FROM gold.fct_orders

    """)

    return tuple(result[0])


before = get_gold_snapshot()

print("\nSTEP 1: GOLD METRICS BEFORE RE-EXECUTION")
print("-" * 55)

print("Total Orders:", before[0])
print("Unique Orders:", before[1])
print("Total Revenue:", before[2])
print("Loyalty Points:", before[3])


# STEP 2: Re-execute the existing Gold transformation

print("\nSTEP 2: RE-EXECUTING GOLD TRANSFORMATION")
print("-" * 55)

run_sql(SQL_40_GOLD)

print("Gold transformation executed again.")


# STEP 3: Capture metrics after re-execution

after = get_gold_snapshot()

print("\nSTEP 3: GOLD METRICS AFTER RE-EXECUTION")
print("-" * 55)

print("Total Orders:", after[0])
print("Unique Orders:", after[1])
print("Total Revenue:", after[2])
print("Loyalty Points:", after[3])


# STEP 4: Compare both snapshots

print("\nSTEP 4: IDEMPOTENCY VALIDATION")
print("-" * 55)

if before == after:

    print("BEFORE AND AFTER METRICS ARE IDENTICAL.")
    print("NO UNEXPECTED CHANGES DETECTED.")
    print("\nFRESHCART IDEMPOTENCY TEST PASSED!")

else:

    print("IDEMPOTENCY TEST FAILED.")
    print("Gold metrics changed after re-execution.")

    raise AssertionError(
        "Gold transformation is not producing identical metrics."
    )


print("\n" + "=" * 75)
print("STAGE 7.1 COMPLETED")
print("=" * 75)

FRESHCART - IDEMPOTENCY TESTING

STEP 1: GOLD METRICS BEFORE RE-EXECUTION
-------------------------------------------------------
Total Orders: 4
Unique Orders: 4
Total Revenue: 7785.00
Loyalty Points: 161

STEP 2: RE-EXECUTING GOLD TRANSFORMATION
-------------------------------------------------------
Gold transformation executed again.

STEP 3: GOLD METRICS AFTER RE-EXECUTION
-------------------------------------------------------
Total Orders: 4
Unique Orders: 4
Total Revenue: 7785.00
Loyalty Points: 161

STEP 4: IDEMPOTENCY VALIDATION
-------------------------------------------------------
BEFORE AND AFTER METRICS ARE IDENTICAL.
NO UNEXPECTED CHANGES DETECTED.

FRESHCART IDEMPOTENCY TEST PASSED!

STAGE 7.1 COMPLETED


| Quantity | Your prediction | Actual after run |
|---|---:|---:|
| Rows for CUST-02 in `dim.customers_scd2` | 2 | 2 |
| Tier used for ORD-1002 | Gold | Gold |
| Loyalty points for ORD-1002 | 70 | 70 |
| Total loyalty points in Gold | 161 | 161 |
| Number of Gold rows whose `published_at` changed | 1 | 1 |

The `row_hash` represents the important values of each record in the Gold table. The `MERGE` statement matches incoming records with existing records using the order ID and updates a record only when its `row_hash` changes. When the Gold transformation is executed again without any changes in the source data, the hashes remain identical, preventing unnecessary updates and maintaining the existing `published_at` timestamps. When Arjun's tier changes from Silver to Gold, the loyalty points for ORD-1002 change, resulting in a different `row_hash` and an updated `published_at` timestamp. Bronze continues to grow by 8 rows on every extraction because each run appends 6 web records and 2 mobile records, while deduplication is performed in the downstream Silver layer.

# B5

In [92]:
# ============================================================
# B5 - STEP 1: VERIFY GOLD TABLE STRUCTURE
# ============================================================

print("=" * 70)
print("VERIFYING GOLD TABLE")
print("=" * 70)

with conn.cursor() as cur:

    # Confirm database context
    cur.execute("SELECT CURRENT_DATABASE()")
    database = cur.fetchone()[0]

    print("Current database:", database)

    # Inspect Gold table columns
    cur.execute("""
        DESCRIBE TABLE FRESHCART.GOLD.FCT_ORDERS
    """)

    columns = cur.fetchall()

    print("\nGOLD.FCT_ORDERS STRUCTURE:\n")

    for column in columns:
        print(
            f"Column: {column[0]:20} "
            f"Type: {column[1]:15} "
            f"Nullable: {column[3]}"
        )

    # Check existing mock ID
    cur.execute("""
        SELECT COUNT(*)
        FROM FRESHCART.GOLD.FCT_ORDERS
        WHERE ORDER_ID = 'B5_MOCK_NEGATIVE_LOYALTY'
    """)

    mock_count = cur.fetchone()[0]

    print("\nExisting mock records:", mock_count)

    # Verify original Gold record count
    cur.execute("""
        SELECT COUNT(*)
        FROM FRESHCART.GOLD.FCT_ORDERS
    """)

    gold_count = cur.fetchone()[0]

    print("Current Gold records:", gold_count)

print("\nTABLE VERIFICATION COMPLETED!")

VERIFYING GOLD TABLE
Current database: FRESHCART

GOLD.FCT_ORDERS STRUCTURE:

Column: ORDER_ID             Type: VARCHAR(16777216) Nullable: N
Column: ORDER_DATE           Type: DATE            Nullable: Y
Column: CUST_ID              Type: VARCHAR(16777216) Nullable: Y
Column: EMAIL                Type: VARCHAR(16777216) Nullable: Y
Column: TIER                 Type: VARCHAR(16777216) Nullable: Y
Column: MULTIPLIER           Type: NUMBER(38,0)    Nullable: Y
Column: AMOUNT               Type: NUMBER(12,2)    Nullable: Y
Column: CURRENCY             Type: VARCHAR(16777216) Nullable: Y
Column: FX_RATE              Type: NUMBER(12,4)    Nullable: Y
Column: AMOUNT_INR           Type: NUMBER(14,2)    Nullable: Y
Column: LOYALTY_POINTS       Type: NUMBER(38,0)    Nullable: Y
Column: IS_HIGH_VALUE        Type: BOOLEAN         Nullable: Y
Column: CITY                 Type: VARCHAR(16777216) Nullable: Y
Column: STATE                Type: VARCHAR(16777216) Nullable: Y
Column: REGION            

In [96]:
# ============================================================
# B5 - STEP 2: INSERT MOCK ORDER WITH NEGATIVE LOYALTY POINTS
# ============================================================

print("=" * 70)
print("B5 - CREATING MOCK ORDER")
print("=" * 70)

MOCK_ORDER_ID = 'B5_MOCK_NEGATIVE_LOYALTY'

with conn.cursor() as cur:

    # Copy an existing Gold record and modify only two values.
    cur.execute("""
        INSERT INTO FRESHCART.GOLD.FCT_ORDERS (
            ORDER_ID,
            ORDER_DATE,
            CUST_ID,
            EMAIL,
            TIER,
            MULTIPLIER,
            AMOUNT,
            CURRENCY,
            FX_RATE,
            AMOUNT_INR,
            LOYALTY_POINTS,
            IS_HIGH_VALUE,
            CITY,
            STATE,
            REGION,
            UPDATED_AT,
            ROW_HASH,
            PUBLISHED_AT
        )
        SELECT
            'B5_MOCK_NEGATIVE_LOYALTY',
            ORDER_DATE,
            CUST_ID,
            EMAIL,
            TIER,
            MULTIPLIER,
            AMOUNT,
            CURRENCY,
            FX_RATE,
            AMOUNT_INR,
            -1,
            IS_HIGH_VALUE,
            CITY,
            STATE,
            REGION,
            UPDATED_AT,
            ROW_HASH,
            PUBLISHED_AT
        FROM FRESHCART.GOLD.FCT_ORDERS
        WHERE ORDER_ID <> 'B5_MOCK_NEGATIVE_LOYALTY'
        LIMIT 1
    """)

    # Verify mock order
    cur.execute("""
        SELECT ORDER_ID, LOYALTY_POINTS
        FROM FRESHCART.GOLD.FCT_ORDERS
        WHERE ORDER_ID = 'B5_MOCK_NEGATIVE_LOYALTY'
    """)

    mock_record = cur.fetchone()

    if mock_record is None:
        raise RuntimeError("Mock order was not inserted.")

    print("\nMOCK ORDER CREATED SUCCESSFULLY!")
    print("Order ID:", mock_record[0])
    print("Loyalty points:", mock_record[1])

    assert mock_record[1] == -1

    # Verify total Gold records
    cur.execute("""
        SELECT COUNT(*)
        FROM FRESHCART.GOLD.FCT_ORDERS
    """)

    print("Total Gold records:", cur.fetchone()[0])

print("\nSTEP 2 COMPLETED!")

B5 - CREATING MOCK ORDER

MOCK ORDER CREATED SUCCESSFULLY!
Order ID: B5_MOCK_NEGATIVE_LOYALTY
Loyalty points: -1
Total Gold records: 5

STEP 2 COMPLETED!


In [97]:
quality_checks()

FRESHCART - FINAL DATA QUALITY VALIDATION
01. ROW COUNT PRESERVATION                       Failed rows: 0     PASSED
02. DUPLICATE ORDERS IN GOLD                     Failed rows: 0     PASSED
03. ORDERS WITHOUT MATCHING CUSTOMER             Failed rows: 0     PASSED
04. ORDERS WITHOUT EXCHANGE RATE                 Failed rows: 0     PASSED
05. ORDERS WITHOUT CITY INFORMATION              Failed rows: 0     PASSED
06. NEGATIVE LOYALTY POINTS IN GOLD              Failed rows: 1     FAILED



AssertionError: FreshCart data quality validation failed.

In [98]:
# ============================================================
# B5 - STEP 3: TEST NEGATIVE LOYALTY POINTS CHECK
# ============================================================

print("=" * 75)
print("B5 - TESTING NEGATIVE LOYALTY POINTS VALIDATION")
print("=" * 75)

with conn.cursor() as cur:

    cur.execute("""
        SELECT COUNT(*)
        FROM FRESHCART.GOLD.FCT_ORDERS
        WHERE LOYALTY_POINTS < 0
    """)

    failed_rows = cur.fetchone()[0]

if failed_rows > 0:
    print("\nPROBLEM DETECTED!")
    print("Check: NEGATIVE LOYALTY POINTS IN GOLD")
    print("Failed rows:", failed_rows)
    print("Status: PROBLEM")
else:
    print("\nNo negative loyalty points found.")
    print("Status: PASSED")

assert failed_rows == 1, (
    f"Expected exactly 1 negative loyalty-point record, found {failed_rows}"
)

print("\nSTEP 3 COMPLETED: NEGATIVE LOYALTY POINTS CHECK DETECTED THE MOCK ORDER.")

B5 - TESTING NEGATIVE LOYALTY POINTS VALIDATION

PROBLEM DETECTED!
Check: NEGATIVE LOYALTY POINTS IN GOLD
Failed rows: 1
Status: PROBLEM

STEP 3 COMPLETED: NEGATIVE LOYALTY POINTS CHECK DETECTED THE MOCK ORDER.


In [99]:
# ============================================================
# B5 - STEP 4: REMOVE MOCK ORDER
# ============================================================

print("=" * 70)
print("B5 - CLEANING UP MOCK ORDER")
print("=" * 70)

with conn.cursor() as cur:

    cur.execute("""
        DELETE FROM FRESHCART.GOLD.FCT_ORDERS
        WHERE ORDER_ID = 'B5_MOCK_NEGATIVE_LOYALTY'
    """)

    # Verify deletion
    cur.execute("""
        SELECT COUNT(*)
        FROM FRESHCART.GOLD.FCT_ORDERS
        WHERE ORDER_ID = 'B5_MOCK_NEGATIVE_LOYALTY'
    """)

    remaining_mock_records = cur.fetchone()[0]

    # Verify original Gold record count
    cur.execute("""
        SELECT COUNT(*)
        FROM FRESHCART.GOLD.FCT_ORDERS
    """)

    gold_count = cur.fetchone()[0]

assert remaining_mock_records == 0, "Mock order was not deleted!"
assert gold_count == 4, f"Expected 4 Gold records, found {gold_count}"

print("\nMOCK ORDER DELETED SUCCESSFULLY!")
print("Remaining mock records:", remaining_mock_records)
print("Total Gold records:", gold_count)

print("\nOriginal Gold data restored successfully!")

B5 - CLEANING UP MOCK ORDER

MOCK ORDER DELETED SUCCESSFULLY!
Remaining mock records: 0
Total Gold records: 4

Original Gold data restored successfully!


In [100]:
quality_checks()

FRESHCART - FINAL DATA QUALITY VALIDATION
01. ROW COUNT PRESERVATION                       Failed rows: 0     PASSED
02. DUPLICATE ORDERS IN GOLD                     Failed rows: 0     PASSED
03. ORDERS WITHOUT MATCHING CUSTOMER             Failed rows: 0     PASSED
04. ORDERS WITHOUT EXCHANGE RATE                 Failed rows: 0     PASSED
05. ORDERS WITHOUT CITY INFORMATION              Failed rows: 0     PASSED
06. NEGATIVE LOYALTY POINTS IN GOLD              Failed rows: 0     PASSED

ALL SIX DATA QUALITY CHECKS PASSED!


[('01. ROW COUNT PRESERVATION', 0),
 ('02. DUPLICATE ORDERS IN GOLD', 0),
 ('03. ORDERS WITHOUT MATCHING CUSTOMER', 0),
 ('04. ORDERS WITHOUT EXCHANGE RATE', 0),
 ('05. ORDERS WITHOUT CITY INFORMATION', 0),
 ('06. NEGATIVE LOYALTY POINTS IN GOLD', 0)]

ELT is a better choice for FreshCart because Snowflake provides scalable computing power to transform data directly inside the data warehouse using SQL. In our pipeline, raw data is first loaded into the Bronze layer, then cleaned and deduplicated in Silver, and finally enriched and stored in Gold. Keeping the original raw data allows us to re-run transformations whenever business rules change or errors are discovered, without extracting the data again from the source systems. However, ETL would be a better choice when sensitive information, such as Aadhaar numbers or credit card details, must be masked or removed before loading data into the warehouse for compliance and security reasons.